# 🚀 Crack-Distill: Mask-KD from SAM 2 (Box Prompts) (Seed 0)
Task loss + Bernoulli Mask-KL to cached SAM 2 Large logits (T=3.7769, W=0.9612).

### Recipe Specifications:
* **Student Model**: YOLOv11n-seg (2.84M parameters, 10.2 GFLOPs)
* **Teacher Model**: SAM 2 Large (`sam2_hiera_large.pt`, 224M parameters)
* **Prompt Type**: Bounding Box Only (offline pre-computed soft logits)
* **Active KD Losses**: Mask-KL ($W=0.9612$)
* **Distillation Temperature**: $\tau = 3.7769$
* **Head Freezing**: Disabled (Full end-to-end training)
* **Precision**: FP32 (`amp: false`) for 100% loss stability
* **Random Seed**: `0`
* **Automated Evaluation**: Evaluates in-domain cropped val and out-of-distribution uncropped val upon completion.


In [1]:
# ── Environment & Directory Initialization ──
!mkdir -p configs utils distillation inference scripts checkpoints data/datasets data/teacher_logits_box runs results
!pip install -q ultralytics==8.4.60 albumentations pycocotools thop pyyaml pandas tqdm opencv-python Pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.9 MB/s eta 0:00:00


In [2]:
%%writefile configs/config.yaml
# ============================================================
# Crack-Distill — Master Config
# ============================================================
# YOU ONLY NEED TO CHANGE:
#   1. data.datasets[0].path  → your crack500_yolo folder
#   2. student.backbone       → yolo11n-seg (demo) or yolo11s-seg (paper)
#   3. train.epochs           → 10 (demo) or 100 (real)
# Everything else leave as is.
# ============================================================

project:
  name: crack-distill
  seed: 42
  output_dir: runs/

# ----------------------------------------------------------
# TASK
# To switch: change type ONLY — everything else adapts.
# Options: instance_seg | semantic_seg | detection
# ----------------------------------------------------------
task:
  type: instance_seg
  num_classes: 1
  class_names:
    - crack

# ----------------------------------------------------------
# DATASET  ← change path here
# ----------------------------------------------------------
data:
  root: data/datasets/
  train_split: 0.8
  val_split: 0.1
  test_split: 0.1
  image_size: 512
  batch_size: 16
  num_workers: 4

  datasets:
    - name: combined
      path: data/datasets/combined_yolo
      format: yolo

# ----------------------------------------------------------
# TEACHER (SAM 2)  ← no changes needed
# ----------------------------------------------------------
teacher:
  model: sam2
  checkpoint: checkpoints/sam2_hiera_large.pt
  config: configs/sam2.1/sam2.1_hiera_l.yaml  # sam2 package config
  device: cuda
  prompt_type: box
  save_logits: true
  logits_dir: data/teacher_logits/
  batch_size: 4

# ----------------------------------------------------------
# STUDENT
# ----------------------------------------------------------
student:
  backbone: yolo11n-seg   # primary student backbone (YOLOv11)
  pretrained: true
  device: "0,1"
  imgsz: 512

# ----------------------------------------------------------
# KD LOSSES  ← no changes needed
# L = L_task + α·L_mask_kd + β·L_feature + γ·L_boundary
# ----------------------------------------------------------
distillation:
  enabled: true
  temperature: 3.7769      # tuned via composite OOD-aware Optuna search

  progressive:
    enabled: true
    stage1_pct: 0.30
    stage2_pct: 0.70

  losses:
    task:
      weight: 1.0
    mask_kd:
      enabled: true
      weight: 0.9612       # tuned KL divergence mask weight
    feature:
      enabled: true
      weight: 1.8658       # tuned intermediate feature alignment weight
      method: cwd          # Channel-Wise Distillation with spatial softmax
      temperature: 4.0
      layers: [16, 19, 22] # true YOLOv11 neck outputs (P3, P4, P5) feeding Segment head
    boundary:
      enabled: true
      weight: 0.8055       # reduced to prevent over-memorizing crop boundary artifacts

# ----------------------------------------------------------
# TRAINING  ← change epochs here
# ----------------------------------------------------------
train:
  epochs: 150    # 10 for demo, 150 for real training
  lr: 0.001
  lr_scheduler: cosine
  warmup_epochs: 3
  optimizer: AdamW
  weight_decay: 0.0005
  amp: false     # disabled FP16 AMP to prevent loss NaN / FP16 overflow on thin crack datasets (e.g. DeepCrack)
  grad_clip: 10.0
  save_every: 10
  patience: 20

# ----------------------------------------------------------
# EVALUATION  ← no changes needed
# ----------------------------------------------------------
eval:
  primary_metric: mAP50-seg
  metrics:
    - mAP50-seg
    - mAP50-95-seg
    - dice
    - boundary_iou
    - fps
    - latency_ms
    - params_M
    - gflops

# ----------------------------------------------------------
# PAPER EXPERIMENTS  ← no changes needed
# ----------------------------------------------------------
experiments:
  - name: baseline_finetune
    distillation.enabled: false
  - name: pseudo_labels
    distillation.losses.mask_kd.enabled: false
    distillation.losses.feature.enabled: false
    distillation.losses.boundary.enabled: false
  - name: full_kd_box
    distillation.enabled: true
    teacher.logits_dir: data/teacher_logits_box/
  - name: full_kd_centroid
    distillation.enabled: true
    teacher.logits_dir: data/teacher_logits_centroid/
  - name: low_data_5pct
    data.train_fraction: 0.05
  - name: low_data_10pct
    data.train_fraction: 0.10
  - name: low_data_25pct
    data.train_fraction: 0.25
  - name: low_data_50pct
    data.train_fraction: 0.50


Writing configs/config.yaml


In [3]:
%%writefile utils/__init__.py
# utils package
from utils.checkpoint import resolve_checkpoint, get_checkpoint_manifest, save_checkpoint_manifest


Writing utils/__init__.py


In [4]:
%%writefile utils/config_loader.py
"""Config loader — converts YAML to a dot-access object."""

import yaml
from pathlib import Path


class ConfigNode:
    """Dot-access config object. cfg.data.batch_size just works."""

    def __init__(self, d: dict):
        for k, v in d.items():
            if isinstance(v, dict):
                setattr(self, k, ConfigNode(v))
            elif isinstance(v, list):
                setattr(self, k, [
                    ConfigNode(i) if isinstance(i, dict) else i for i in v
                ])
            else:
                setattr(self, k, v)

    def get(self, key, default=None):
        return getattr(self, key, default)

    def keys(self):
        return self.__dict__.keys()

    def values(self):
        return self.__dict__.values()

    def items(self):
        return self.__dict__.items()

    def __getitem__(self, key):
        if hasattr(self, key):
            return getattr(self, key)
        raise KeyError(key)

    def __setitem__(self, key, value):
        setattr(self, key, value)

    def __contains__(self, key):
        return hasattr(self, key)

    def __repr__(self):
        return f"ConfigNode({self.__dict__})"

    def __iter__(self):
        return iter(self.__dict__.keys())

    def dict(self):
        result = {}
        for k, v in self.__dict__.items():
            if isinstance(v, ConfigNode):
                result[k] = v.dict()
            elif isinstance(v, list):
                result[k] = [i.dict() if isinstance(i, ConfigNode) else i for i in v]
            else:
                result[k] = v
        return result


def load_config(path: str) -> ConfigNode:
    """Load YAML config and return dot-access ConfigNode."""
    with open(path) as f:
        raw = yaml.safe_load(f)
    return ConfigNode(raw)


def override_config(cfg: ConfigNode, overrides: dict) -> ConfigNode:
    """
    Apply flat-key overrides to a config.
    e.g. override_config(cfg, {"distillation.enabled": False})
    """
    raw = cfg.dict()
    for key_path, value in overrides.items():
        parts = key_path.split(".")
        node = raw
        for p in parts[:-1]:
            node = node.setdefault(p, {})
        node[parts[-1]] = value
    return ConfigNode(raw)


Writing utils/config_loader.py


In [5]:
%%writefile utils/checkpoint.py
"""
Checkpoint provenance and deterministic resolution utility.
Addresses scientific audit findings:
- Eliminates non-deterministic glob-based checkpoint collision.
- Records immutable SHA256 manifests for every trained model.
"""

import hashlib
import json
import os
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Union


def compute_file_sha256(file_path: Union[str, Path], chunk_size: int = 1024 * 1024) -> str:
    """
    Compute streaming SHA256 hash of a file.

    Args:
        file_path (str | Path): Path to file.
        chunk_size (int): Read chunk size in bytes (default: 1MB).

    Returns:
        str: Hex-encoded SHA256 digest.
    """
    path = Path(file_path)
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint file not found: {path}")

    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()


def resolve_checkpoint(
    checkpoint_path: Union[str, Path],
    expected_experiment: Optional[str] = None,
    allow_search: bool = False,
) -> Path:
    """
    Deterministically resolve a checkpoint file path without fuzzy wildcards.

    Args:
        checkpoint_path (str | Path): File path or directory containing weights.
        expected_experiment (str, optional): Expected experiment name for sanity checking.
        allow_search (bool): If True and path is directory, checks standard weights/best.pt.

    Returns:
        Path: Verified checkpoint path.

    Raises:
        FileNotFoundError: If checkpoint does not exist.
        ValueError: If file is not a valid PyTorch weight file or empty.
    """
    path = Path(checkpoint_path)

    # 1. Direct file resolution
    if path.is_file():
        if path.suffix not in (".pt", ".pth", ".bin"):
            raise ValueError(f"Checkpoint file has unsupported extension '{path.suffix}': {path}")
        if path.stat().st_size == 0:
            raise ValueError(f"Checkpoint file is empty (0 bytes): {path}")
        return path.resolve()

    # 2. Directory resolution
    if path.is_dir():
        candidates = [
            path / "weights" / "best.pt",
            path / "best.pt",
            path / "weights" / "last.pt",
            path / "last.pt",
        ]
        for candidate in candidates:
            if candidate.is_file() and candidate.stat().st_size > 0:
                return candidate.resolve()

        if expected_experiment:
            named_candidates = [
                path / expected_experiment / "weights" / "best.pt",
                path / "segment" / expected_experiment / "weights" / "best.pt",
                path / "crack_distill" / expected_experiment / "weights" / "best.pt",
            ]
            for candidate in named_candidates:
                if candidate.is_file() and candidate.stat().st_size > 0:
                    return candidate.resolve()

    raise FileNotFoundError(
        f"Could not deterministically resolve checkpoint from path '{checkpoint_path}'. "
        f"Expected existing .pt file or directory containing weights/best.pt."
    )


def get_checkpoint_manifest(
    checkpoint_path: Union[str, Path],
    experiment_name: Optional[str] = None,
    seed: Optional[int] = None,
    metadata: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Build immutable provenance record for a checkpoint file.

    Returns dict with sha256, file size, timestamp, experiment, seed, and optional metadata.
    """
    resolved_path = resolve_checkpoint(checkpoint_path)
    stat = resolved_path.stat()
    sha256 = compute_file_sha256(resolved_path)

    manifest: Dict[str, Any] = {
        "checkpoint_path": str(resolved_path),
        "filename": resolved_path.name,
        "sha256": sha256,
        "size_bytes": stat.st_size,
        "modified_time_iso": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(stat.st_mtime)),
        "experiment": experiment_name,
        "seed": seed,
        "metadata": metadata or {},
    }
    return manifest


def save_checkpoint_manifest(manifest: Dict[str, Any], output_path: Optional[Union[str, Path]] = None) -> Path:
    """
    Write checkpoint manifest to JSON file.
    Defaults to <checkpoint_dir>/<checkpoint_stem>_manifest.json.
    """
    if output_path is None:
        ckpt_p = Path(manifest["checkpoint_path"])
        output_path = ckpt_p.parent / f"{ckpt_p.stem}_manifest.json"

    out_p = Path(output_path)
    out_p.parent.mkdir(parents=True, exist_ok=True)
    with open(out_p, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    return out_p


def discover_and_deduplicate_checkpoints(
    search_roots: Optional[List[Union[str, Path]]] = None,
    patterns: tuple = ("**/best.pt",),
) -> List[Dict[str, Any]]:
    """
    Search directory trees for checkpoints, resolving paths and deduplicating
    by SHA256 file digest to prevent duplicate evaluations or collision.

    Returns a list of dicts:
        [{
            "name": str,       # Disambiguated experiment name
            "path": Path,      # Resolved checkpoint file path
            "sha256": str,     # Full SHA256 digest
            "short_sha": str,  # First 8 characters of SHA256
            "size_bytes": int, # Size in bytes
        }, ...]
    """
    if search_roots is None:
        search_roots = [Path("/kaggle/input"), Path("runs")]

    raw_paths: List[Path] = []
    for root in search_roots:
        root_p = Path(root)
        if root_p.exists():
            for pat in patterns:
                raw_paths.extend(root_p.glob(pat))

    seen_hashes: Dict[str, Path] = {}
    discovered: List[Dict[str, Any]] = []

    for p in sorted(raw_paths):
        try:
            resolved = resolve_checkpoint(p)
        except Exception:
            continue

        sha = compute_file_sha256(resolved)
        if sha in seen_hashes:
            continue

        seen_hashes[sha] = resolved
        parts = resolved.parts
        if len(parts) >= 3 and parts[-2] == "weights":
            exp_name = parts[-3]
        elif len(parts) >= 2:
            exp_name = parts[-2]
        else:
            exp_name = resolved.stem

        # Disambiguate name if already present
        base_name = exp_name
        counter = 1
        existing_names = [d["name"] for d in discovered]
        while exp_name in existing_names:
            exp_name = f"{base_name}_{counter}"
            counter += 1

        discovered.append({
            "name": exp_name,
            "path": resolved,
            "sha256": sha,
            "short_sha": sha[:8],
            "size_bytes": resolved.stat().st_size,
        })

    return discovered


Writing utils/checkpoint.py


In [6]:
%%writefile inference/__init__.py
# inference package
from inference.tiled_inference import tiled_predict_image_gaussian, create_gaussian_weight_map


Writing inference/__init__.py


In [7]:
%%writefile inference/tiled_inference.py
"""
CrackDistill — Gaussian-Weighted Tiled Sliding-Window Inference Engine
=====================================================================
Performs overlapping sliding-window inference on high-resolution images
(e.g., 2000x1500 full-scene asphalt pavement photographs) using 2D Gaussian
apodization blending to eliminate tiling boundary discontinuities.
"""

from pathlib import Path
import cv2
import numpy as np
import torch


def create_gaussian_weight_map(tile_size: int = 512, sigma: float = 0.35) -> np.ndarray:
    """
    Generate a 2D Gaussian window to smoothly blend overlapping inference tiles.

    Args:
        tile_size (int): Dimension of square tile (default 512).
        sigma (float): Gaussian spread parameter relative to [-1, 1] range (default 0.35).

    Returns:
        np.ndarray: Normalized 2D float32 weight array of shape (tile_size, tile_size).
    """
    ax = np.linspace(-1, 1, tile_size)
    gauss_1d = np.exp(-0.5 * (ax / sigma) ** 2)
    gauss_2d = np.outer(gauss_1d, gauss_1d).astype(np.float32)
    gauss_2d = np.maximum(gauss_2d, 0.05)  # Minimum floor to prevent edge zero division
    return gauss_2d / gauss_2d.max()


def tiled_predict_prob_map(
    model,
    img_bgr: np.ndarray,
    tile_size: int = 512,
    stride: int = 384,
    conf: float = 0.25,
    sigma: float = 0.35,
    device: str | None = None,
) -> np.ndarray:
    """
    Run overlapping sliding-window inference on full-resolution image with
    2D Gaussian apodization blending, returning continuous probability map.

    Args:
        model: Ultralytics YOLO segmentation model or callable.
        img_bgr (np.ndarray): Full-resolution input image in BGR format (H, W, 3).
        tile_size (int): Size of sliding-window tile (default 512).
        stride (int): Sliding step in pixels; overlap is tile_size - stride (default 384 = 25% overlap).
        conf (float): Detection confidence threshold for model.predict (default 0.25).
        sigma (float): 2D Gaussian apodization parameter (default 0.35).
        device (str | None): Computing device ('cuda', 'cpu', or auto-detect if None).

    Returns:
        np.ndarray: Continuous crack probability map of shape (H, W) in range [0.0, 1.0].
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    h, w = img_bgr.shape[:2]
    full_prob_map = np.zeros((h, w), dtype=np.float32)
    weight_accum_map = np.zeros((h, w), dtype=np.float32)
    weight_window = create_gaussian_weight_map(tile_size, sigma)

    # Compute step coordinates covering the entire canvas
    y_steps = list(range(0, max(1, h - tile_size + 1), stride))
    if not y_steps or y_steps[-1] + tile_size < h:
        y_steps.append(max(0, h - tile_size))

    x_steps = list(range(0, max(1, w - tile_size + 1), stride))
    if not x_steps or x_steps[-1] + tile_size < w:
        x_steps.append(max(0, w - tile_size))

    for y0 in y_steps:
        for x0 in x_steps:
            tile = img_bgr[y0 : y0 + tile_size, x0 : x0 + tile_size]
            
            # Pad tile if canvas is smaller than tile_size
            pad_h = max(0, tile_size - tile.shape[0])
            pad_w = max(0, tile_size - tile.shape[1])
            if pad_h > 0 or pad_w > 0:
                tile = cv2.copyMakeBorder(tile, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)

            results = model.predict(tile, imgsz=tile_size, conf=conf, verbose=False, device=device)
            r = results[0]

            tile_prob = np.zeros((tile_size, tile_size), dtype=np.float32)
            if r.masks is not None and len(r.masks) > 0:
                for m in r.masks.data.cpu().numpy():
                    m_resized = cv2.resize(m, (tile_size, tile_size), interpolation=cv2.INTER_LINEAR)
                    tile_prob = np.maximum(tile_prob, m_resized)

            # Unpad if tile was padded
            actual_h = min(tile_size, h - y0)
            actual_w = min(tile_size, w - x0)

            full_prob_map[y0 : y0 + actual_h, x0 : x0 + actual_w] += (
                tile_prob[:actual_h, :actual_w] * weight_window[:actual_h, :actual_w]
            )
            weight_accum_map[y0 : y0 + actual_h, x0 : x0 + actual_w] += (
                weight_window[:actual_h, :actual_w]
            )

    weight_accum_map = np.maximum(weight_accum_map, 1e-5)
    return full_prob_map / weight_accum_map


def tiled_predict_image_gaussian(
    model,
    img_bgr: np.ndarray,
    tile_size: int = 512,
    stride: int = 384,
    conf: float = 0.25,
    threshold: float = 0.35,
    sigma: float = 0.35,
    device: str | None = None,
) -> np.ndarray:
    """
    Run overlapping sliding-window inference on full-resolution image with
    2D Gaussian apodization blending, returning binary mask (0 or 1).

    Args:
        model: Ultralytics YOLO segmentation model or callable.
        img_bgr (np.ndarray): Full-resolution input image in BGR format (H, W, 3).
        tile_size (int): Size of sliding-window tile (default 512).
        stride (int): Sliding step in pixels (default 384).
        conf (float): Detection confidence threshold (default 0.25).
        threshold (float): Final binarization probability threshold (default 0.35).
        sigma (float): 2D Gaussian apodization parameter (default 0.35).
        device (str | None): Computing device ('cuda', 'cpu', or auto-detect).

    Returns:
        np.ndarray: Binary crack segmentation mask of shape (H, W) with values {0, 1}, uint8.
    """
    prob_map = tiled_predict_prob_map(
        model=model,
        img_bgr=img_bgr,
        tile_size=tile_size,
        stride=stride,
        conf=conf,
        sigma=sigma,
        device=device,
    )
    return (prob_map > threshold).astype(np.uint8)


def benchmark_tiled_inference_pipeline(
    model,
    scene_shape: tuple = (1500, 2000, 3),
    tile_size: int = 512,
    stride: int = 384,
    n_runs: int = 5,
    device: str | None = None,
    image_shape: tuple | None = None,
    overlap: float | None = None,
    num_runs: int | None = None,
) -> dict:
    """
    Benchmark and disambiguate single-tile forward latency vs full-scene tiled inference pipeline.
    Addresses Problem P2-5:
    - Level 1: Single-Tile Forward Latency (512x512).
    - Level 2: Full-Scene Tiled Reconstruction Pipeline (2000x1500, 20 tiles, 2D Gaussian apodization blending).
    """
    import time

    if image_shape is not None:
        scene_shape = image_shape
    if num_runs is not None:
        n_runs = num_runs
    if overlap is not None:
        stride = max(32, int(tile_size * (1.0 - overlap)))

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Synthetic scene for profiling
    dummy_scene = np.random.randint(0, 256, scene_shape, dtype=np.uint8)
    dummy_tile = np.random.randint(0, 256, (tile_size, tile_size, 3), dtype=np.uint8)

    def _safe_predict(input_data):
        try:
            return model.predict(input_data, imgsz=tile_size, verbose=False, device=device)
        except TypeError:
            return model.predict(input_data, imgsz=tile_size, verbose=False)

    # 1. Warm-up
    for _ in range(3):
        _ = _safe_predict(dummy_tile)

    # 2. Single-Tile Forward Latency
    single_tile_times = []
    for _ in range(25):
        t0 = time.perf_counter()
        _ = _safe_predict(dummy_tile)
        if device == "cuda":
            torch.cuda.synchronize()
        single_tile_times.append((time.perf_counter() - t0) * 1000)

    tile_ms = float(np.mean(single_tile_times))
    tile_fps = 1000.0 / tile_ms if tile_ms > 0 else 0.0

    # 3. Full-Scene Tiled Inference Pipeline (2000x1500)
    h, w = scene_shape[:2]
    y_steps = list(range(0, max(1, h - tile_size + 1), stride))
    if not y_steps or y_steps[-1] + tile_size < h:
        y_steps.append(max(0, h - tile_size))
    x_steps = list(range(0, max(1, w - tile_size + 1), stride))
    if not x_steps or x_steps[-1] + tile_size < w:
        x_steps.append(max(0, w - tile_size))
    n_tiles = len(y_steps) * len(x_steps)

    scene_times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = tiled_predict_image_gaussian(model, dummy_scene, tile_size=tile_size, stride=stride, device=device)
        if device == "cuda":
            torch.cuda.synchronize()
        scene_times.append((time.perf_counter() - t0) * 1000)

    scene_ms = float(np.mean(scene_times))
    scene_fps = 1000.0 / scene_ms if scene_ms > 0 else 0.0

    return {
        "device": device,
        "single_tile": {
            "resolution": f"{tile_size}x{tile_size}",
            "mean_latency_ms": tile_ms,
            "throughput_fps": tile_fps,
        },
        "full_scene": {
            "resolution": f"{w}x{h}",
            "num_tiles": n_tiles,
            "mean_pipeline_ms": scene_ms,
            "scenes_per_second": scene_fps,
            "pipeline_includes": "sliding_window_tiling + model_predict + 2d_gaussian_blending + thresholding",
        },
        "num_tiles": n_tiles,
        "level1_single_tile": {
            "mean_ms": tile_ms,
            "fps": tile_fps,
        },
        "pipeline_serial": {
            "mean_ms": scene_ms,
            "fps": scene_fps,
        },
        "critical_disambiguation_note": (
            f"Level 1: Single-Tile forward pass = {tile_ms:.2f} ms ({tile_fps:.1f} FPS). "
            f"Level 2: Full-Scene ({w}x{h}, {n_tiles} tiles) reconstruction = {scene_ms:.1f} ms ({scene_fps:.1f} scenes/sec)."
        )
    }



Writing inference/tiled_inference.py


In [8]:
%%writefile distillation/__init__.py
# distillation package


Writing distillation/__init__.py


In [9]:
%%writefile distillation/kd_trainer.py
"""
KD Trainer — correct implementation
=====================================
Processes soft logits and intermediate encoder features from SAM.
Registers hooks and trains 1x1 projection convolutions for feature distillation.
Uses picklable hooks and temporary hook/loss restoration during model saving to prevent pickling errors.
"""

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from collections import OrderedDict
from ultralytics.models.yolo.segment.train import SegmentationTrainer


# Spatial augmentations zeroed for controlled comparison (P0-3); shared with scripts/verify_alignment.py
SPATIAL_AUG_OFF = {
    "mosaic": 0.0,
    "close_mosaic": 0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "fliplr": 0.0,
    "flipud": 0.0,
    "erasing": 0.0,
}

def teacher_kind(logits_dir) -> str:
    """Which teacher a logits directory holds, from its name: 'gt_soft', 'centroid' or 'sam_box'."""
    name = Path(str(logits_dir).rstrip("/")).name.lower()
    if "gt_soft" in name:
        return "gt_soft"
    if "centroid" in name:
        return "centroid"
    return "sam_box"


def letterbox_content_box(orig_hw, ratio_pad, imgsz, target_hw):
    """
    Locate the real image content of a YOLO letterbox canvas inside a (target_h, target_w) grid.

    Single source of truth for teacher -> student coordinate warping (P0-1), shared by the
    mask/feature KD losses, scripts/verify_alignment.py and tests/test_coordinate_warping.py.

    Args:
        orig_hw: (h, w) of the source image the SAM teacher saw, or None if unknown.
        ratio_pad: Ultralytics batch ratio_pad, or None. Two formats occur:
            - eval (LetterBox as a direct transform): ((h_ratio, w_ratio), (left, top))
            - training (augment=True): (h_ratio, w_ratio) only, with no padding entry even
              though the canvas is still centre-letterboxed, so padding is reconstructed.
            With rect=False and scaleup=False the LetterBox gain is 1, so the ratios are the
            full source -> canvas scale.
        imgsz: (H, W) of the letterbox canvas.
        target_hw: (h, w) of the grid the loss is computed on.

    Returns:
        (top, left, unpad_h, unpad_w) in target pixels, or None when orig_hw is unknown.
    """
    if orig_hw is None:
        return None
    orig_h, orig_w = float(orig_hw[0]), float(orig_hw[1])
    target_h, target_w = int(target_hw[0]), int(target_hw[1])

    if ratio_pad is not None and isinstance(ratio_pad[0], (tuple, list)):
        (r_h, r_w), (pad_w, pad_h) = ratio_pad
        r_h, r_w, pad_w, pad_h = float(r_h), float(r_w), float(pad_w), float(pad_h)
    else:
        if ratio_pad is not None:
            r_h, r_w = float(ratio_pad[0]), float(ratio_pad[1])
        else:
            r_h = r_w = min(float(imgsz[0]) / orig_h, float(imgsz[1]) / orig_w)
        # Centre padding, as LetterBox(center=True) applies it
        pad_w = (float(imgsz[1]) - round(orig_w * r_w)) / 2.0
        pad_h = (float(imgsz[0]) - round(orig_h * r_h)) / 2.0

    scale_y = target_h / float(imgsz[0])
    scale_x = target_w / float(imgsz[1])
    top = max(0, min(int(round(pad_h * scale_y)), target_h - 1))
    left = max(0, min(int(round(pad_w * scale_x)), target_w - 1))
    unpad_h = max(1, min(int(round(orig_h * r_h * scale_y)), target_h - top))
    unpad_w = max(1, min(int(round(orig_w * r_w * scale_x)), target_w - left))
    return top, left, unpad_h, unpad_w


def warp_to_letterbox(tensor, box, target_hw, pad_value):
    """
    Resize a source-space (N, C, h, w) tensor onto the content box and pad the letterbox border.

    Args:
        tensor: (N, C, h, w) teacher logits or features spanning the full source image.
        box: (top, left, unpad_h, unpad_w) from letterbox_content_box().
        target_hw: (h, w) of the output grid.
        pad_value: fill value for the letterbox border (e.g. -20.0 background logit).
    """
    top, left, unpad_h, unpad_w = box
    bottom = max(0, int(target_hw[0]) - top - unpad_h)
    right = max(0, int(target_hw[1]) - left - unpad_w)
    unpad = F.interpolate(tensor, size=(unpad_h, unpad_w), mode="bilinear", align_corners=False)
    return F.pad(unpad, (left, right, top, bottom), mode="constant", value=pad_value)


class KDYOLODataset(torch.utils.data.Dataset):
    """
    Wrapper for YOLO Dataset that preloads SAM teacher logits and features
    inside dataloader worker processes to hide disk I/O latency from GPU training.
    """
    def __init__(self, base_dataset, logits_dir, kd_cfg):
        self.base_dataset = base_dataset
        self.logits_dir = Path(logits_dir)
        self.features_dir = self.logits_dir.parent / "teacher_features"
        self.kd_cfg = kd_cfg

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        img_path = item.get("im_file", "")
        if not img_path:
            item["sam_target"] = None
            item["sam_feat"] = None
            return item
            
        stem = Path(img_path).stem

        sam_target = None
        sam_feat = None

        for prefix in [f"crack500_{stem}", f"deepcrack_{stem}", stem]:
            c = self.logits_dir / f"{prefix}_logits.npy"
            f_path = self.features_dir / f"{prefix}_features.npz"

            if c.exists():
                try:
                    raw = np.load(str(c))  # shape (M, 256, 256)
                    if raw.ndim == 2:
                        raw = np.expand_dims(raw, axis=0)
                    sam_target = torch.from_numpy(raw).float()
                except Exception:
                    pass

                if self.kd_cfg.losses.feature.enabled:
                    if not f_path.exists():
                        f_path = self.logits_dir / f"{prefix}_features.npz"
                    if f_path.exists():
                        try:
                            with np.load(str(f_path)) as data:
                                sam_feat = {
                                    "image_embed": torch.from_numpy(data["image_embed"]).float(),
                                    "feat1": torch.from_numpy(data["feat1"]).float()
                                }
                                if "feat0" in data:
                                    sam_feat["feat0"] = torch.from_numpy(data["feat0"]).float()
                        except Exception:
                            pass
                break

        item["sam_target"] = sam_target
        item["sam_feat"] = sam_feat
        return item

    @property
    def collate_fn(self):
        return self.base_dataset.collate_fn

    def __getattr__(self, name):
        return getattr(self.base_dataset, name)


class ActiveHook:
    """
    A top-level picklable hook callback class.
    Writes features directly to a class attribute to avoid referencing local closures.
    """
    def __init__(self, key):
        self.key = key

    def __call__(self, module, input, output):
        KDSegmentationTrainer.student_features[self.key] = output


class KDSegmentationTrainer(SegmentationTrainer):
    # Class-level attribute to store hooked features safely
    student_features = {}

    def __init__(self, cfg=None, overrides=None, _callbacks=None, logits_dir=None, kd_cfg=None, **kwargs):
        import os
        from pathlib import Path
        from ultralytics.cfg import DEFAULT_CFG

        if cfg is None:
            cfg = DEFAULT_CFG

        # Auto-convert master ConfigNode / master dict into Ultralytics cfg if master config passed directly
        if hasattr(cfg, "student") or (isinstance(cfg, dict) and "student" in cfg):
            master_cfg = cfg
            if kd_cfg is None:
                if hasattr(master_cfg, "distillation"):
                    kd_cfg = master_cfg.distillation
                elif isinstance(master_cfg, dict) and "distillation" in master_cfg:
                    kd_cfg = master_cfg["distillation"]

            if logits_dir is None:
                if hasattr(master_cfg, "teacher") and hasattr(master_cfg.teacher, "logits_dir"):
                    logits_dir = master_cfg.teacher.logits_dir
                elif isinstance(master_cfg, dict) and "teacher" in master_cfg and "logits_dir" in master_cfg["teacher"]:
                    logits_dir = master_cfg["teacher"]["logits_dir"]

            # Backbone model
            model_name = "yolo11n-seg"
            if hasattr(master_cfg, "student") and hasattr(master_cfg.student, "backbone"):
                model_name = master_cfg.student.backbone
            elif isinstance(master_cfg, dict) and "student" in master_cfg and "backbone" in master_cfg["student"]:
                model_name = master_cfg["student"]["backbone"]
            if not str(model_name).endswith(".pt"):
                model_name = f"{model_name}.pt"

            # Data YAML path
            data_path = "data/datasets/crack500_yolo/dataset.yaml"
            if hasattr(master_cfg, "data"):
                if hasattr(master_cfg.data, "datasets") and len(master_cfg.data.datasets) > 0:
                    d_p = master_cfg.data.datasets[0].path
                    candidate = d_p if str(d_p).endswith(".yaml") else os.path.join(d_p, "dataset.yaml")
                    if os.path.exists(candidate):
                        data_path = candidate
                    elif os.path.exists("data/datasets/crack500_yolo/dataset.yaml"):
                        data_path = "data/datasets/crack500_yolo/dataset.yaml"
                elif hasattr(master_cfg.data, "path"):
                    d_p = master_cfg.data.path
                    data_path = d_p if str(d_p).endswith(".yaml") else os.path.join(d_p, "dataset.yaml")
            elif isinstance(master_cfg, dict) and "data" in master_cfg:
                d_dict = master_cfg["data"]
                if "datasets" in d_dict and len(d_dict["datasets"]) > 0:
                    d_p = d_dict["datasets"][0].get("path", data_path) if isinstance(d_dict["datasets"][0], dict) else getattr(d_dict["datasets"][0], "path", data_path)
                    data_path = d_p if str(d_p).endswith(".yaml") else os.path.join(d_p, "dataset.yaml")

            check_data = (overrides.get("data") if overrides and isinstance(overrides, dict) else None) or data_path
            if check_data and not Path(check_data).exists():
                # Check local alternatives
                for alt_candidate in [
                    Path("data/datasets/combined_yolo/dataset.yaml"),
                    Path("data/datasets/crack500_yolo/dataset.yaml"),
                ]:
                    if alt_candidate.exists():
                        check_data = str(alt_candidate)
                        data_path = check_data
                        break

                # Auto-detect datasets in Kaggle input mount (/kaggle/input/**/dataset.yaml)
                if not Path(check_data).exists() and Path("/kaggle/input").exists():
                    kaggle_yamls = sorted(list(Path("/kaggle/input").glob("**/dataset.yaml")))
                    if kaggle_yamls:
                        # Prefer crack500 or combined or distill datasets
                        preferred = [y for y in kaggle_yamls if any(k in str(y).lower() for k in ["crack500", "combined", "distill"])]
                        chosen_yaml = preferred[0] if preferred else kaggle_yamls[0]
                        check_data = str(chosen_yaml)
                        data_path = check_data
                        print(f"[KD] Auto-detected Kaggle dataset: {chosen_yaml}")

                if overrides and isinstance(overrides, dict) and "data" in overrides:
                    overrides["data"] = check_data

            # Safely fix hardcoded absolute paths in dataset.yaml.
            # The dataset folder may be a symlink to a read-only /kaggle/input path,
            # so we NEVER write to the symlink target.
            # Instead, we copy dataset.yaml to a writable local path and use that copy.
            if check_data and Path(check_data).exists():
                try:
                    import shutil as _shutil
                    yaml_p = Path(check_data)
                    # Resolve the ACTUAL target to check if it is read-only
                    real_yaml = yaml_p.resolve()
                    abs_dset_dir = str(yaml_p.parent.resolve())
                    original_text = real_yaml.read_text()
                    fixed_lines = [f"path: {abs_dset_dir}" if l.strip().startswith("path:") else l for l in original_text.splitlines()]
                    fixed_text = "\n".join(fixed_lines) + "\n"
                    # Only write if the text actually changed
                    if fixed_text != original_text:
                        # Try writing in place (works if writable)
                        try:
                            yaml_p.write_text(fixed_text)
                            print(f"[KD] Fixed dataset.yaml path in-place: {abs_dset_dir}")
                        except OSError:
                            # Read-only filesystem (Kaggle input symlink): copy to local writable dir
                            local_yaml_dir = Path("data/datasets_yaml")
                            local_yaml_dir.mkdir(parents=True, exist_ok=True)
                            local_yaml = local_yaml_dir / yaml_p.parent.name / "dataset.yaml"
                            local_yaml.parent.mkdir(parents=True, exist_ok=True)
                            local_yaml.write_text(fixed_text)
                            data_path = str(local_yaml)
                            if overrides and isinstance(overrides, dict) and "data" in overrides:
                                overrides["data"] = str(local_yaml)
                            print(f"[KD] Read-only symlink — copied fixed dataset.yaml to: {local_yaml}")
                except Exception as e:
                    print(f"[KD Warning] dataset.yaml path fix skipped: {e}")

            proj_name = getattr(getattr(master_cfg, "project", None), "name", "runs")
            exp_name = getattr(getattr(master_cfg, "project", None), "experiment", "exp")
            seed_val = int(getattr(getattr(master_cfg, "project", None), "seed", 42))

            # Enforce determinism
            import random
            random.seed(seed_val)
            np.random.seed(seed_val)
            torch.manual_seed(seed_val)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed_val)

            auto_overrides = {
                "model": model_name,
                "data": data_path,
                "epochs": getattr(getattr(master_cfg, "train", None), "epochs", 150),
                "imgsz": getattr(getattr(master_cfg, "student", None), "imgsz", getattr(getattr(master_cfg, "data", None), "image_size", 512)),
                "batch": getattr(getattr(master_cfg, "data", None), "batch_size", 16),
                "amp": getattr(getattr(master_cfg, "train", None), "amp", False),
                "lr0": getattr(getattr(master_cfg, "train", None), "lr", 0.001),
                "weight_decay": getattr(getattr(master_cfg, "train", None), "weight_decay", 0.0005),
                "project": str(proj_name),
                "name": str(exp_name),
                "seed": seed_val,
                "exist_ok": True,
                "task": "segment",
                # Teacher logits are stored in label-file order and paired via teacher[target_gt_idx];
                # overlap masks re-sort batch instances by area. Set for both arms to keep parity.
                "overlap_mask": False,
            }

            # Scientific Fix (P0-3): Enforce identical augmentation policy across all experimental arms.
            # To eliminate the augmentation confound, both baseline and KD runs must adhere to the same policy.
            # Spatial scrambling (mosaic, affine, flip) without matching teacher transforms corrupts supervision coordinates.
            allow_spatial_aug = getattr(getattr(master_cfg, "train", None), "allow_spatial_aug", False)
            if not allow_spatial_aug:
                auto_overrides.update(SPATIAL_AUG_OFF)
                print(f"[Augmentation Policy] Spatial augmentations (mosaic, affine, flip, erasing) disabled for controlled comparison (allow_spatial_aug=False). Seed: {seed_val}")

            if overrides and isinstance(overrides, dict):
                auto_overrides.update(overrides)

            from ultralytics.cfg import get_cfg
            cfg = get_cfg(overrides=auto_overrides)
            overrides = {}

        super().__init__(cfg=cfg, overrides=overrides, _callbacks=_callbacks, **kwargs)

        if logits_dir is None:
            logits_dir = os.environ.get("KD_LOGITS_DIR", "data/teacher_logits/")
                
        if kd_cfg is None:
            kd_config_json = os.environ.get("KD_CONFIG")
            if kd_config_json:
                try:
                    import json
                    from utils.config_loader import ConfigNode
                    kd_cfg = ConfigNode(json.loads(kd_config_json))
                except Exception:
                    pass
            
            if kd_cfg is None:
                # Under DDP, we can load configuration dynamically as a fallback
                from utils.config_loader import load_config
                try:
                    full_cfg = load_config("configs/config.yaml")
                    kd_cfg = full_cfg.distillation
                except Exception:
                    pass

        self.logits_dir  = Path(str(logits_dir))
        self.kd_cfg      = kd_cfg

        # Scientific Fix (P0-2): Strict baseline control arm gating.
        # When distillation.enabled is False, cleanly bypass all teacher logit checking,
        # projection layers, hooks, loss patching, and dataset wrapping.
        is_kd_on = False
        if kd_cfg is not None:
            if hasattr(kd_cfg, "enabled"):
                is_kd_on = bool(getattr(kd_cfg, "enabled", False))
            elif isinstance(kd_cfg, dict):
                is_kd_on = bool(kd_cfg.get("enabled", False))
        self.is_kd_on = is_kd_on

        # KD pairs batch instance k with teacher instance k (label-file order); overlap masks
        # re-sort instances by area, so enforce per-instance masks whichever way we were built.
        if self.is_kd_on and getattr(self.args, "overlap_mask", False):
            print("[KD] overlap_mask=False enforced: overlap masks re-sort instances by area and break teacher pairing.")
            self.args.overlap_mask = False

        if kd_cfg is not None and hasattr(kd_cfg, "temperature"):
            self.temperature = float(kd_cfg.temperature)
        else:
            self.temperature = 1.6502
        
        self._current_paths = []
        self._kd_logged  = False
        self._kd_consecutive_errors = 0
        self._kd_total_errors = 0
        self._kd_total_batches = 0
        self._dropped_instances_count = 0
        self._total_instances_count = 0
        self._no_logits_warned = False
        self._geom_fallback_warned = False
        self._kd_unmatched_batches = 0
        self._sam_targets = {}   # image_stem → soft target tensor (M, 256, 256)
        self._sam_features = {}  # image_stem → dict of features
        self._hook_handles = []

        if not self.is_kd_on:
            self.proj_layers = torch.nn.ModuleDict()
            print("[KD] Distillation DISABLED (`distillation.enabled=False`). Initialized Clean Baseline Control (native YOLOv11).")
            return

        logit_files = list(self.logits_dir.glob("*.npy"))
        if len(logit_files) == 0:
            candidates = [
                Path("/tmp") / self.logits_dir.name,
                Path("/tmp/teacher_logits_box"),
                Path("/tmp/teacher_logits"),
            ]
            if Path("/kaggle/input").exists():
                candidates.extend([
                    Path("/kaggle/input/teacher-logits-box/teacher_logits"),
                    Path("/kaggle/input/teacher_logits_box/teacher_logits"),
                    Path("/kaggle/input/teacher-logits-box"),
                    Path("/kaggle/input/teacher_logits_box"),
                    Path("/kaggle/input/teacher_logits"),
                ])
                for d in Path("/kaggle/input").glob("**/*logits*"):
                    if d.is_dir() and d not in candidates:
                        candidates.append(d)

            wanted_kind = teacher_kind(self.logits_dir)
            for candidate in candidates:
                # Never substitute a different teacher (e.g. SAM logits for the GT-soft control arm)
                if teacher_kind(candidate) != wanted_kind:
                    continue
                if candidate.exists():
                    c_files = list(candidate.glob("*.npy"))
                    if len(c_files) > 0:
                        print(f"[KD] Found {len(c_files)} logit files in fallback '{candidate}'. Redirecting logits_dir.")
                        self.logits_dir = candidate
                        logit_files = c_files
                        try:
                            import shutil
                            p_local = Path(str(logits_dir))
                            if not p_local.exists() or (p_local.is_dir() and not list(p_local.glob("*.npy"))):
                                if os.path.lexists(p_local):
                                    os.unlink(p_local) if os.path.islink(p_local) else shutil.rmtree(p_local)
                                p_local.parent.mkdir(parents=True, exist_ok=True)
                                os.symlink(candidate, p_local)
                                print(f"[KD] Created symlink: {p_local} -> {candidate}")
                        except Exception as sym_err:
                            print(f"[KD Warning] Could not symlink fallback: {sym_err}")
                        break

        print(f"[KD] logits_dir : {self.logits_dir}")
        print(f"[KD] logit files: {len(logit_files)}")
        print(f"[KD] temperature: {self.temperature}")

        if len(logit_files) == 0:
            raise FileNotFoundError(
                f"[KD FATAL ERROR] logits_dir '{self.logits_dir}' contains 0 logit files (*.npy)!\n"
                f"Knowledge distillation cannot proceed without precomputed teacher logits.\n"
                f"Please ensure the teacher logits dataset is attached and linked properly to '{self.logits_dir}'."
            )

    def get_instance_drop_stats(self):
        """Return diagnostic statistics on dropped teacher instance masks."""
        total = self._total_instances_count
        dropped = self._dropped_instances_count
        rate = (dropped / max(1, total)) if total > 0 else 0.0
        return {
            "total_instances": total,
            "dropped_instances": dropped,
            "drop_rate": rate,
        }

    def _letterbox_box(self, orig_hw, ratio_pad, imgsz, target_hw, stem):
        """
        letterbox_content_box() with a loud failure when the batch lacks ori_shape.

        Without it the teacher can only be plain-resized over the padded canvas, which is the
        pre-P0-1 misalignment. Strict mode (default) raises; permissive mode warns once.
        """
        box = letterbox_content_box(orig_hw, ratio_pad, imgsz, target_hw)
        if box is None:
            msg = (
                f"Missing letterbox metadata (ori_shape) for '{stem}': the teacher cannot be aligned to the "
                f"letterboxed student canvas, and a plain resize would reintroduce the P0-1 padding offset."
            )
            if getattr(self.kd_cfg, "strict", True) if self.kd_cfg is not None else True:
                raise RuntimeError(msg)
            if not self._geom_fallback_warned:
                print(f"[KD Warning] {msg} Falling back to plain resize (strict=False).")
                self._geom_fallback_warned = True
        return box

    def setup_model(self):
        """Build model, set up projection layers and hooks, and call parent setup."""
        if not self.is_kd_on:
            print("[KD] Clean Baseline Control: Native YOLO setup (hooks, projection layers, and loss patch bypassed) ✓")
            return super().setup_model()

        head_idx = 23
        is_freeze_head = hasattr(self.kd_cfg, "progressive") and self.kd_cfg.progressive.get("freeze_head", False)

        if is_freeze_head:
            if self.args.freeze is None:
                self.args.freeze = [head_idx]
            elif isinstance(self.args.freeze, list):
                if head_idx not in self.args.freeze:
                    self.args.freeze.append(head_idx)
            elif isinstance(self.args.freeze, int):
                self.args.freeze = list(range(self.args.freeze))
                if head_idx not in self.args.freeze:
                    self.args.freeze.append(head_idx)
            print(f"[KD] Progressive: Freezing Segment head at index {head_idx}. args.freeze={self.args.freeze}")

        ckpt = super().setup_model()

        if is_freeze_head:
            try:
                from ultralytics.utils.torch_utils import unwrap_model
                model = unwrap_model(self.model)
            except Exception:
                model = self.model

            for idx, module in enumerate(model.model):
                if type(module).__name__ == "Segment":
                    head_idx = idx
                    break

            for name, param in model.named_parameters():
                if f"model.{head_idx}." in name:
                    param.requires_grad = False
            print(f"[KD] Explicitly set requires_grad=False for Segment head parameters (layer {head_idx})")

        self._setup_proj_layers_and_hooks()
        self._patch_model_loss()
        return ckpt

    def _setup_proj_layers_and_hooks(self):
        """
        Dynamically determine student backbone feature shapes, initialize 1x1 convs
        for channel alignment, and register active training hooks.
        """
        try:
            from ultralytics.utils.torch_utils import unwrap_model
            model = unwrap_model(self.model)
        except Exception:
            model = self.model

        # Determine device
        device = next(self.model.parameters()).device

        # Standard layers to monitor: YOLOv11 true neck outputs feeding Segment head are 16 (P3, stride 8), 19 (P4, stride 16), 22 (P5, stride 32)
        feat_method = getattr(self.kd_cfg.losses.feature, "method", "mse") if hasattr(self.kd_cfg.losses, "feature") else "mse"
        default_layers = [16, 19, 22]
        layers_to_monitor = list(self.kd_cfg.losses.feature.layers) if hasattr(self.kd_cfg.losses.feature, "layers") else default_layers
        if layers_to_monitor in ([12, 15, 18], [2, 5, 8], [4, 6, 8]):
            print(f"[KD] Corrected legacy hooked layers {layers_to_monitor} -> [16, 19, 22] (true P3, P4, P5 neck outputs)")
            layers_to_monitor = [16, 19, 22]

        captured_shapes = {}
        def temp_hook(layer_idx):
            def hook(module, input, output):
                captured_shapes[layer_idx] = output.shape
            return hook

        hooks = []
        for idx in layers_to_monitor:
            if idx < len(model.model):
                h = model.model[idx].register_forward_hook(temp_hook(idx))
                hooks.append(h)

        # Run dummy forward pass to extract shapes
        dummy_input = torch.zeros((1, 3, self.args.imgsz, self.args.imgsz), device=device)
        model.eval()
        with torch.no_grad():
            try:
                _ = model(dummy_input)
            except Exception as e:
                print(f"[KD] Error during dummy forward pass for shapes: {e}")
        model.train()

        # Remove temporary hooks
        for h in hooks:
            h.remove()

        # Build projection layers
        layer_target_map = {
            16: ("feat1", 64),
            19: ("image_embed", 256),
            22: ("image_embed_p5", 256),
        }
        proj_dict = nn.ModuleDict()
        for idx in layers_to_monitor:
            if idx in captured_shapes:
                in_channels = captured_shapes[idx][1]
                feature_h = captured_shapes[idx][2]
                if idx in layer_target_map:
                    target_key, out_channels = layer_target_map[idx]
                    stride = 8 if target_key == "feat1" else (16 if target_key == "image_embed" else 32)
                else:
                    img_sz = self.args.imgsz if isinstance(self.args.imgsz, int) else self.args.imgsz[0]
                    stride = img_sz // feature_h
                    if stride <= 4:
                        out_channels = 32
                    elif stride <= 8:
                        out_channels = 64
                    else:
                        out_channels = 256
                
                # Cross-Architecture Projector (CAP): 2-stage conv with BatchNorm for CWD, or 1x1 for MSE
                if feat_method == "cwd":
                    proj_dict[f"layer_{idx}"] = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                        nn.BatchNorm2d(out_channels),
                        nn.GELU(),
                        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, groups=out_channels, bias=False),
                        nn.BatchNorm2d(out_channels)
                    )
                else:
                    proj_dict[f"layer_{idx}"] = nn.Conv2d(in_channels, out_channels, kernel_size=1)
                print(f"[KD] Feature projection layer {idx} ({feat_method}): stride {stride}, channels {in_channels} -> {out_channels}")

        # Register projection layers on the model so they are part of optimizer parameters
        model.add_module("proj_layers", proj_dict)
        self.proj_layers = proj_dict.to(device)

        # Register active training hooks
        self._register_active_hooks(model)

    def _register_active_hooks(self, model):
        """Helper to register forward hooks on target student model layers."""
        self._hook_handles.clear()
        KDSegmentationTrainer.student_features.clear()
        
        feat_method = getattr(self.kd_cfg.losses.feature, "method", "mse") if hasattr(self.kd_cfg.losses, "feature") else "mse"
        default_layers = [16, 19, 22]
        layers_to_monitor = list(self.kd_cfg.losses.feature.layers) if hasattr(self.kd_cfg.losses.feature, "layers") else default_layers
        if layers_to_monitor in ([12, 15, 18], [2, 5, 8], [4, 6, 8]):
            layers_to_monitor = [16, 19, 22]

        for idx in layers_to_monitor:
            if idx < len(model.model):
                h = model.model[idx].register_forward_hook(ActiveHook(f"layer_{idx}"))
                self._hook_handles.append(h)
                print(f"[KD] Forward hook registered for layer {idx}")

    def save_model(self):
        """Override save_model to temporarily detach hooks and restore original loss function on both model and EMA model."""
        if not self.is_kd_on:
            return super().save_model()

        try:
            from ultralytics.utils.torch_utils import unwrap_model
            model = unwrap_model(self.model)
        except Exception:
            model = self.model

        # Get EMA model if defined
        ema_model = None
        if hasattr(self, "ema") and self.ema is not None and hasattr(self.ema, "ema"):
            try:
                ema_model = unwrap_model(self.ema.ema)
            except Exception:
                ema_model = self.ema.ema

        # Remove hooks on self.model
        for handle in self._hook_handles:
            handle.remove()
        self._hook_handles.clear()

        # Recursively clear forward hooks in all submodules for both models
        for m in [model, ema_model]:
            if m is not None:
                for submodule in m.modules():
                    submodule._forward_hooks.clear()

        # Restore original loss function if patched on both models
        original_loss_restored = False
        for m in [model, ema_model]:
            if m is not None and hasattr(m, "original_loss"):
                m.loss = m.original_loss
                original_loss_restored = True

        # Call original saving logic
        result = super().save_model()

        # Re-patch loss function on training model
        if original_loss_restored:
            self._patch_model_loss()

        # Re-register active hooks on training model
        self._register_active_hooks(model)
        
        # Reset the active hooks registered flag so that they get registered on the active training model again
        self._active_hooks_registered = False
        return result

    def build_dataset(self, img_path: str, mode: str = "train", batch: int | None = None):
        """Build custom KD dataset that wraps the default YOLO dataset."""
        base_dataset = super().build_dataset(img_path, mode, batch)
        if mode != "train" or not self.is_kd_on:
            return base_dataset
        return KDYOLODataset(base_dataset, self.logits_dir, self.kd_cfg)

    def preprocess_batch(self, batch):
        """Preprocess batch and map preloaded SAM targets/features to GPU."""
        if not self.is_kd_on:
            return super().preprocess_batch(batch)

        # Ensure active hooks are registered on the active running model (handles DDP deepcopy recreation)
        if not hasattr(self, "_active_hooks_registered") or not self._active_hooks_registered:
            try:
                from ultralytics.utils.torch_utils import unwrap_model
                active_model = unwrap_model(self.model)
            except Exception:
                active_model = self.model
            self._register_active_hooks(active_model)
            self._active_hooks_registered = True

        # Clear student features at the start of batch preprocessing
        KDSegmentationTrainer.student_features.clear()

        # Safety check to verify that all projection layer parameters are in the optimizer
        if not hasattr(self, "_checked_optimizer") and hasattr(self, "optimizer") and self.optimizer is not None:
            self._checked_optimizer = True
            proj_params = set(self.proj_layers.parameters())
            opt_params = set()
            for group in self.optimizer.param_groups:
                for p in group['params']:
                    opt_params.add(p)
            missing = proj_params - opt_params
            if missing:
                print(f"[KD] WARNING: {len(missing)} projection layer parameters are NOT in the optimizer! Training them will have no effect.")
            else:
                print("[KD] Success: All projection layer parameters are in the optimizer and will receive gradients.")

        batch = super().preprocess_batch(batch)
        im_files = batch.get("im_file", [])
        if isinstance(im_files, (str, Path)):
            im_files = [im_files]
        self._current_paths = list(im_files)
        self._current_ratio_pad = batch.get("ratio_pad", None)
        self._current_ori_shape = batch.get("ori_shape", None)

        # Retrieve the preloaded SAM targets and features from the batch dict
        sam_targets_list = batch.get("sam_target", [])
        sam_feats_list = batch.get("sam_feat", [])

        self._sam_targets = {}
        self._sam_features = {}

        for idx, img_path in enumerate(self._current_paths):
            stem = Path(img_path).stem
            
            if idx < len(sam_targets_list) and sam_targets_list[idx] is not None:
                # Clean NaNs and Infs to prevent nan mask_kd losses
                self._sam_targets[stem] = torch.nan_to_num(sam_targets_list[idx].to(self.device), nan=0.0, posinf=0.0, neginf=0.0)
                
            if idx < len(sam_feats_list) and sam_feats_list[idx] is not None:
                self._sam_features[stem] = {
                    k: torch.nan_to_num(v.to(self.device), nan=0.0, posinf=0.0, neginf=0.0) for k, v in sam_feats_list[idx].items()
                }

        if self._current_paths and not self._sam_targets:
            self._kd_unmatched_batches += 1
            if not self._no_logits_warned:
                stems = [Path(p).stem for p in self._current_paths[:3]]
                print(f"[KD] Warning: no SAM logits matched batch stems {stems}. "
                      f"Run: python scripts/generate_teacher_logits.py")
                self._no_logits_warned = True
            # A KD run whose teacher files never match silently degenerates into the baseline
            if self._kd_unmatched_batches >= 10 and (getattr(self.kd_cfg, "strict", True) if self.kd_cfg is not None else True):
                raise RuntimeError(
                    f"[KD] No teacher logits in '{self.logits_dir}' matched {self._kd_unmatched_batches} consecutive "
                    f"training batches (expected '<stem>_logits.npy'). Refusing to train KD without a teacher."
                )
        else:
            self._kd_unmatched_batches = 0

        return batch

    def _patch_model_loss(self):
        """
        Dynamically patch Ultralytics Segment model loss function with multi-component KD losses.
        """
        if not self.is_kd_on:
            return

        trainer_ref = self

        try:
            from ultralytics.utils.torch_utils import unwrap_model
            model = unwrap_model(self.model)
        except Exception:
            model = self.model

        if not hasattr(model, "original_loss"):
            model.original_loss = model.loss

        original_loss_fn = model.original_loss.__func__ if hasattr(model.original_loss, "__func__") else None

        def patched_loss(self_model, batch, preds=None):
            if preds is None:
                preds = self_model.forward(batch["img"])

            if original_loss_fn is not None:
                base_loss, loss_items = original_loss_fn(self_model, batch, preds)
            else:
                base_loss, loss_items = type(self_model).loss(self_model, batch, preds)

            if not self_model.training:
                return base_loss, loss_items

            kd_losses = trainer_ref._kd_loss_from_preds(preds, batch, self_model)
            
            # Combine losses
            total = base_loss
            for k, v in kd_losses.items():
                total = total + v

            return total, loss_items

        import types
        model.loss = types.MethodType(patched_loss, model)
        print("[KD] model.loss() patched with detailed KD losses ✓")

    def _kd_loss_from_preds(self, preds, batch, model) -> dict:
        """
        Compute KL divergence, boundary, and feature alignment losses.
        """
        kd_losses = {}
        if not self.is_kd_on or not self._sam_targets:
            return kd_losses

        try:
            self._kd_total_batches += 1
            criterion = model.criterion
            preds_parsed = criterion.parse_output(preds)
            
            # Retrieve target assignments
            (fg_mask, target_gt_idx, target_bboxes, _, _), _, _ = criterion.get_assigned_targets_and_loss(preds_parsed, batch)
            
            pred_masks = preds_parsed["mask_coefficient"].permute(0, 2, 1).contiguous()
            proto = preds_parsed["proto"]
            imgsz = batch["img"].shape[-2:] if "img" in batch else (512, 512)
            ratio_pad_list = batch.get("ratio_pad", getattr(self, "_current_ratio_pad", None))
            ori_shape_list = batch.get("ori_shape", getattr(self, "_current_ori_shape", None))
            
            loss_mask_kd = torch.tensor(0.0, device=self.device)
            loss_mask_kd_count = 0
            
            loss_affinity = torch.tensor(0.0, device=self.device)
            loss_affinity_count = 0

            loss_boundary = torch.tensor(0.0, device=self.device)
            loss_boundary_count = 0

            loss_tversky = torch.tensor(0.0, device=self.device)
            loss_tversky_count = 0

            # 1. Compute L_mask and L_boundary (Per-instance matched)
            for i, img_path in enumerate(self._current_paths):
                stem = Path(img_path).stem
                if stem not in self._sam_targets:
                    continue
                
                sam_logits = self._sam_targets[stem]
                fg_mask_i = fg_mask[i]
                
                if fg_mask_i.any() and sam_logits.shape[0] > 0:
                    raw_mask_idx = target_gt_idx[i][fg_mask_i]

                    # Scientific Fix (Bug #2 & P1-1): Strict instance index validation with diagnostic tracking
                    n_inst = raw_mask_idx.numel()
                    valid_idx_mask = (raw_mask_idx >= 0) & (raw_mask_idx < sam_logits.shape[0])
                    n_valid = int(valid_idx_mask.sum().item())
                    n_dropped = n_inst - n_valid
                    self._total_instances_count += n_inst
                    self._dropped_instances_count += n_dropped

                    if n_dropped > 0:
                        print(
                            f"[KD Warning] Dropped {n_dropped}/{n_inst} teacher instances for '{stem}' "
                            f"(indices out of bounds [0, {sam_logits.shape[0] - 1}]). "
                            f"Cumulative dropped instances: {self._dropped_instances_count}/{self._total_instances_count}"
                        )

                    if not valid_idx_mask.any():
                        continue

                    mask_idx = raw_mask_idx[valid_idx_mask]
                    pred_coefs = pred_masks[i][fg_mask_i][valid_idx_mask]
                    pred_mask_logits = torch.einsum("in,nhw->ihw", pred_coefs, proto[i])
                    sam_logits_matched = sam_logits[mask_idx]
                    
                    # Target resolution (default 256x256, or 512x512 if high_res, or native teacher res if native_res enabled)
                    if getattr(self.kd_cfg.losses.mask_kd, "native_res", False) or getattr(self.kd_cfg.losses.mask_kd, "upsample_student", False):
                        target_h, target_w = sam_logits_matched.shape[-2:]
                    elif getattr(self.kd_cfg.losses.mask_kd, "high_res", False):
                        target_h, target_w = 512, 512
                    else:
                        target_h, target_w = 256, 256
                    
                    # Resize student mask logits to target resolution
                    student_mask_logits_resized = F.interpolate(
                        pred_mask_logits.unsqueeze(1),
                        size=(target_h, target_w),
                        mode="bilinear",
                        align_corners=False
                    ).squeeze(1)

                    # Scientific Fix (P0-1): Exact Letterbox Coordinate Warping
                    # SAM teacher logits span [0, orig_h] x [0, orig_w].
                    # Student proto spans letterboxed imgsz canvas with padding.
                    # Project teacher logits into student canvas with exact padding alignment.
                    orig_hw = ori_shape_list[i] if ori_shape_list is not None and i < len(ori_shape_list) else None
                    ratio_pad_i = ratio_pad_list[i] if ratio_pad_list is not None and i < len(ratio_pad_list) else None
                    box = self._letterbox_box(orig_hw, ratio_pad_i, imgsz, (target_h, target_w), stem)
                    if box is not None:
                        t_top, t_left, t_unpad_h, t_unpad_w = box
                        # Pad canvas with -20.0 (background logit, sigmoid(-20/T) ~ 0)
                        sam_logits_matched_resized = warp_to_letterbox(
                            sam_logits_matched.unsqueeze(1), box, (target_h, target_w), pad_value=-20.0
                        ).squeeze(1)

                        valid_content_mask = torch.zeros(
                            (target_h, target_w),
                            dtype=torch.bool,
                            device=student_mask_logits_resized.device
                        )
                        valid_content_mask[t_top : t_top + t_unpad_h, t_left : t_left + t_unpad_w] = True
                    else:
                        sam_logits_matched_resized = F.interpolate(
                            sam_logits_matched.unsqueeze(1),
                            size=(target_h, target_w),
                            mode="bilinear",
                            align_corners=False
                        ).squeeze(1)
                        valid_content_mask = torch.ones(
                            (target_h, target_w),
                            dtype=torch.bool,
                            device=student_mask_logits_resized.device
                        )
                        t_top, t_left = 0, 0
                        t_unpad_h, t_unpad_w = target_h, target_w

                    # Align dtypes to prevent precision/autocast mismatches
                    sam_logits_matched_resized = sam_logits_matched_resized.to(dtype=student_mask_logits_resized.dtype)
                    
                    # L_mask (KL Divergence on Bernoulli soft probabilities)
                    # FIX: clamp logits before sigmoid to prevent log(0) -> NaN
                    mask_kd_cfg = getattr(getattr(self.kd_cfg, "losses", None), "mask_kd", None)
                    if mask_kd_cfg and getattr(mask_kd_cfg, "enabled", False):
                        sam_clamped = torch.clamp(sam_logits_matched_resized / self.temperature, -15.0, 15.0)
                        stu_clamped = torch.clamp(student_mask_logits_resized / self.temperature, -15.0, 15.0)
                        q = torch.sigmoid(sam_clamped)
                        p_log = F.logsigmoid(stu_clamped)
                        inv_q = 1.0 - q
                        inv_p_log = F.logsigmoid(-stu_clamped)
                        
                        kl = q * (torch.log(q + 1e-8) - p_log) + inv_q * (torch.log(inv_q + 1e-8) - inv_p_log)
                        
                        # Foreground-Dilated / Region-Focused Mask-KL (if enabled)
                        use_focused = getattr(mask_kd_cfg, "focused", False) or getattr(mask_kd_cfg, "foreground_dilated", False)
                        if use_focused:
                            fg_core = (q > 0.35).float().unsqueeze(1)
                            fg_dilated = F.max_pool2d(fg_core, kernel_size=9, stride=1, padding=4).squeeze(1)
                            fg_core = fg_core.squeeze(1)
                            weight_map = torch.where(fg_core > 0, 1.0, torch.where(fg_dilated > 0, 0.5, 0.05))
                            weight_map = weight_map * valid_content_mask.unsqueeze(0).float()
                            kl_weighted = (kl * weight_map).sum(dim=(-1, -2)) / (weight_map.sum(dim=(-1, -2)) + 1e-6)
                            loss_mask_kd = loss_mask_kd + kl_weighted.mean() * (self.temperature ** 2)
                        else:
                            val_mask_f = valid_content_mask.unsqueeze(0).float()
                            kl_valid = (kl * val_mask_f).sum(dim=(-1, -2)) / (val_mask_f.sum(dim=(-1, -2)) + 1e-6)
                            loss_mask_kd = loss_mask_kd + kl_valid.mean() * (self.temperature ** 2)
                        loss_mask_kd_count += 1

                    # L_affinity (Spatial Pixel Affinity / Directional Gradient KD)
                    affinity_cfg = getattr(getattr(self.kd_cfg, "losses", None), "affinity", None)
                    if affinity_cfg and getattr(affinity_cfg, "enabled", False):
                        p_stu = torch.sigmoid(stu_clamped)
                        p_stu_crop = p_stu[:, t_top : t_top + t_unpad_h, t_left : t_left + t_unpad_w]
                        q_crop = q[:, t_top : t_top + t_unpad_h, t_left : t_left + t_unpad_w]
                        if p_stu_crop.shape[-2] > 1 and p_stu_crop.shape[-1] > 1:
                            d_stu_x = p_stu_crop[:, :, 1:] - p_stu_crop[:, :, :-1]
                            d_stu_y = p_stu_crop[:, 1:, :] - p_stu_crop[:, :-1, :]
                            d_tea_x = q_crop[:, :, 1:] - q_crop[:, :, :-1]
                            d_tea_y = q_crop[:, 1:, :] - q_crop[:, :-1, :]
                            loss_aff = F.mse_loss(d_stu_x, d_tea_x.detach()) + F.mse_loss(d_stu_y, d_tea_y.detach())
                            loss_affinity = loss_affinity + loss_aff
                            loss_affinity_count += 1

                    # L_boundary (Per-instance matched boundary weighted loss)
                    boundary_cfg = getattr(getattr(self.kd_cfg, "losses", None), "boundary", None)
                    if boundary_cfg and getattr(boundary_cfg, "enabled", False):
                        sam_soft = torch.sigmoid(sam_logits_matched_resized / self.temperature)
                        bw = (1.0 - torch.abs(sam_soft - 0.5) * 2).detach()
                        stu_clamped_raw = torch.clamp(student_mask_logits_resized, -30.0, 30.0)
                        bce = F.binary_cross_entropy_with_logits(
                            stu_clamped_raw, sam_soft.detach(), reduction="none"
                        )
                        bw_valid = bw * valid_content_mask.unsqueeze(0).float()
                        denom = bw_valid.sum(dim=(-1, -2)).clamp(min=1e-6)
                        loss_boundary = loss_boundary + ((bce * bw_valid).sum(dim=(-1, -2)) / denom).mean()
                        loss_boundary_count += 1

                    # L_tversky (Asymmetric Tversky loss targeting thin-crack continuity)
                    tversky_cfg = getattr(getattr(self.kd_cfg, "losses", None), "tversky", None)
                    if tversky_cfg and getattr(tversky_cfg, "enabled", False):
                        p_stu = torch.sigmoid(stu_clamped)
                        alpha = float(getattr(tversky_cfg, "alpha", 0.30))
                        beta = float(getattr(tversky_cfg, "beta", 0.70))
                        eps = 1.0
                        val_mask_f = valid_content_mask.unsqueeze(0).float()
                        p_stu_val = p_stu * val_mask_f
                        q_val = q * val_mask_f
                        tp = (p_stu_val * q_val).sum(dim=(-1, -2))
                        fp = (p_stu_val * (1.0 - q_val)).sum(dim=(-1, -2))
                        fn = ((1.0 - p_stu_val) * q_val).sum(dim=(-1, -2))
                        tversky_idx = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
                        loss_tversky = loss_tversky + (1.0 - tversky_idx).mean()
                        loss_tversky_count += 1

            if mask_kd_cfg and getattr(mask_kd_cfg, "enabled", False) and loss_mask_kd_count > 0:
                kd_losses["mask_kd"] = (loss_mask_kd / loss_mask_kd_count) * float(getattr(mask_kd_cfg, "weight", 1.0))

            if affinity_cfg and getattr(affinity_cfg, "enabled", False) and loss_affinity_count > 0:
                kd_losses["affinity"] = (loss_affinity / loss_affinity_count) * float(getattr(affinity_cfg, "weight", 1.0))
                
            if boundary_cfg and getattr(boundary_cfg, "enabled", False) and loss_boundary_count > 0:
                kd_losses["boundary"] = (loss_boundary / loss_boundary_count) * float(getattr(boundary_cfg, "weight", 1.0))

            if tversky_cfg and getattr(tversky_cfg, "enabled", False) and loss_tversky_count > 0:
                kd_losses["tversky"] = (loss_tversky / loss_tversky_count) * float(getattr(tversky_cfg, "weight", 1.0))

            # 2. Compute L_feature (Scale-matched alignment or CWD)
            feat_cfg = getattr(getattr(self.kd_cfg, "losses", None), "feature", None)
            if feat_cfg and getattr(feat_cfg, "enabled", False):
                loss_feat = torch.tensor(0.0, device=self.device)
                feat_method = getattr(self.kd_cfg.losses.feature, "method", "mse")
                default_layers = [16, 19, 22]
                layers_to_monitor = list(self.kd_cfg.losses.feature.layers) if hasattr(self.kd_cfg.losses.feature, "layers") else default_layers
                if layers_to_monitor in ([12, 15, 18], [2, 5, 8], [4, 6, 8]):
                    layers_to_monitor = [16, 19, 22]
                feat_count = 0
                
                # Layer importance weights (P3 = 0.5, P4 = 0.3, P5 = 0.2)
                stage_weights = {16: 0.5, 19: 0.3, 22: 0.2}
                
                for idx in layers_to_monitor:
                    feat_key = f"layer_{idx}"
                    if feat_key in self.student_features and feat_key in self.proj_layers:
                        sf = self.student_features[feat_key]
                        proj = self.proj_layers[feat_key]
                        sf_proj = proj(sf)
                        
                        feature_h = sf.shape[2]
                        layer_target_map = {
                            16: ("feat1", 64),
                            19: ("image_embed", 256),
                            22: ("image_embed_p5", 256),
                        }
                        if idx in layer_target_map:
                            target_key, out_channels = layer_target_map[idx]
                        else:
                            img_sz = self.args.imgsz if isinstance(self.args.imgsz, int) else self.args.imgsz[0]
                            stride = img_sz // feature_h
                            if stride <= 4:
                                target_key = "feat0"
                            elif stride <= 8:
                                target_key = "feat1"
                            elif stride <= 16:
                                target_key = "image_embed"
                            else:
                                target_key = "image_embed_p5"
                        # Stack SAM features for the batch (per-item spatial alignment before concat)
                        tf_list = []
                        valid_item_indices = []
                        target_h, target_w = sf_proj.shape[2], sf_proj.shape[3]
                        for b_idx, img_path in enumerate(self._current_paths):
                            stem = Path(img_path).stem
                            has_target = stem in self._sam_features and (
                                target_key in self._sam_features[stem] or
                                (target_key == "image_embed_p5" and "image_embed" in self._sam_features[stem])
                            )
                            if has_target:
                                if target_key in self._sam_features[stem]:
                                    tf_item = self._sam_features[stem][target_key]
                                else:
                                    # Dynamically pool image_embed (stride 16 -> stride 32)
                                    raw_emb = self._sam_features[stem]["image_embed"]
                                    if raw_emb.ndim == 3:
                                        tf_item = F.avg_pool2d(raw_emb.unsqueeze(0), kernel_size=2, stride=2).squeeze(0)
                                    else:
                                        tf_item = F.avg_pool2d(raw_emb, kernel_size=2, stride=2)

                                if tf_item.ndim == 3:
                                    tf_item = tf_item.unsqueeze(0)
                                
                                # Letterbox warping for teacher features (shared P0-1 geometry)
                                f_orig_hw = ori_shape_list[b_idx] if ori_shape_list is not None and b_idx < len(ori_shape_list) else None
                                f_ratio_pad = ratio_pad_list[b_idx] if ratio_pad_list is not None and b_idx < len(ratio_pad_list) else None
                                f_box = self._letterbox_box(f_orig_hw, f_ratio_pad, imgsz, (target_h, target_w), stem)
                                if f_box is not None:
                                    tf_item = warp_to_letterbox(tf_item, f_box, (target_h, target_w), pad_value=0.0)
                                elif tf_item.shape[2:] != (target_h, target_w):
                                    tf_item = F.interpolate(tf_item, size=(target_h, target_w), mode="bilinear", align_corners=False)
                                
                                tf_list.append(tf_item)
                                valid_item_indices.append(b_idx)
                        
                        # Scientific Fix: Only compute feature loss on samples with valid teacher features (no zero-filling uniform noise)
                        if len(valid_item_indices) > 0:
                            tf_batch = torch.cat(tf_list, dim=0).to(dtype=sf_proj.dtype)
                            sf_proj_valid = sf_proj[valid_item_indices]
                            
                            # Ensure channel dimensions match
                            if tf_batch.shape[1] != sf_proj_valid.shape[1]:
                                if not self._no_logits_warned:
                                    print(f"[KD Warning] Channel mismatch for layer {idx}: student proj {sf_proj_valid.shape[1]} vs teacher {tf_batch.shape[1]}. Skipping layer.")
                                continue
                            
                            b_v, c, h, w = sf_proj_valid.shape
                            b_t, c_t, h_t, w_t = tf_batch.shape
                            if (h, w) != (h_t, w_t):
                                tf_batch = F.interpolate(tf_batch, size=(h, w), mode="bilinear", align_corners=False)
                            
                            if feat_method == "cwd":
                                # Channel-Wise Distillation (Spatial Softmax per channel + KL Divergence)
                                t_feat = float(getattr(self.kd_cfg.losses.feature, "temperature", 4.0))
                                s_soft = F.softmax(sf_proj_valid.view(b_v, c, -1) / t_feat, dim=-1)
                                t_soft = F.softmax(tf_batch.detach().view(b_t, c_t, -1) / t_feat, dim=-1)
                                # Scientific Fix: Normalize CWD KL divergence by channel count c
                                cwd_kl = (F.kl_div(s_soft.log(), t_soft, reduction="batchmean") * (t_feat ** 2)) / c
                                w_stage = stage_weights.get(idx, 1.0 / len(layers_to_monitor))
                                loss_feat = loss_feat + w_stage * cwd_kl
                                feat_count += 1
                            else:
                                # Standard normalized per-layer MSE
                                loss_feat = loss_feat + F.mse_loss(sf_proj_valid, tf_batch.detach())
                                feat_count += 1
                    else:
                        if feat_key not in self.student_features and not self._no_logits_warned:
                            print(f"[KD] Warning: Hook feature {feat_key} not found in student_features. "
                                  f"Forward hooks might not be triggering. Skipping feature KD.")
                            self._no_logits_warned = True
                
                if feat_count > 0:
                    if feat_method != "cwd":
                        loss_feat = loss_feat / feat_count
                    kd_losses["feature"] = loss_feat * self.kd_cfg.losses.feature.weight

            # Reset consecutive errors on successful batch computation
            self._kd_consecutive_errors = 0

            # Logging demonstration on first pass
            if not self._kd_logged and kd_losses:
                log_strs = [f"{k}: {float(v.detach()):.6f}" for k, v in kd_losses.items()]
                print(f"[KD] ✓ KD losses computed: {', '.join(log_strs)}")
                self._kd_logged = True

        except Exception as e:
            self._kd_consecutive_errors += 1
            self._kd_total_errors += 1
            strict_kd = getattr(self.kd_cfg, "strict", True) if self.kd_cfg is not None else True
            
            # Fail fast if strict mode is enabled, preventing silent training corruption
            if strict_kd:
                import traceback
                tb = traceback.format_exc()
                raise RuntimeError(
                    f"[KD Critical Error] KD loss computation failed in strict mode (batch failure #{self._kd_total_errors}):\n"
                    f"{type(e).__name__}: {e}\n"
                    f"Traceback:\n{tb}\n"
                    f"Halting training immediately (strict=True) to prevent silent distillation corruption."
                ) from e
            else:
                if self._kd_consecutive_errors > 5:
                    raise RuntimeError(
                        f"[KD Critical Error] KD loss computation failed for {self._kd_consecutive_errors} consecutive batches. "
                        f"Last error ({type(e).__name__}): {e}. Halting training to prevent silent degeneration."
                    ) from e
                elif self._kd_consecutive_errors <= 3 or self._kd_total_errors % 50 == 0:
                    print(f"[KD Warning] Batch KD loss error (consecutive={self._kd_consecutive_errors}, total={self._kd_total_errors}): "
                          f"{type(e).__name__}: {e}")
                    if self._kd_consecutive_errors <= 2:
                        import traceback
                        traceback.print_exc()

        return kd_losses


Writing distillation/kd_trainer.py


In [10]:
%%writefile scripts/convert_crack500.py
#!/usr/bin/env python3
"""
Crack500 → YOLO seg format converter
=====================================
Crack500 structure:
  crack500/
  ├── traincrop/   ← 00001.jpg + 00001.png (binary mask, same stem)
  ├── valcrop/
  ├── testcrop/
  ├── train.txt    ← list of image filenames (optional)
  ├── val.txt
  └── test.txt

Output (YOLO seg format, ready for ultralytics):
  crack500_yolo/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── dataset.yaml

Each .txt label: one line per connected crack instance
  0 x1 y1 x2 y2 ... (normalized polygon, class 0 = crack)

Usage:
  python scripts/convert_crack500.py \
      --src ~/distill/data/datasets/crack500 \
      --dst ~/distill/data/datasets/crack500_yolo
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int) -> list[str]:
    """
    Read binary PNG mask → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.

    Returns list of label lines (one per instance).
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    # Threshold (Crack500 masks are 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path, split_name: str):
    """Process one split (train/val/test)."""

    # Crack500 stores images+masks together in traincrop/valcrop/testcrop
    crop_dir = src_dir / f"{split_name}crop"
    if not crop_dir.exists():
        # Try alternate names
        for candidate in [src_dir / split_name, src_dir / f"{split_name}data"]:
            if candidate.exists():
                crop_dir = candidate
                break
        else:
            print(f"  [WARNING] Could not find directory for split '{split_name}', skipping.")
            return 0

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all images (jpg/jpeg/png that are NOT masks)
    all_files = sorted(crop_dir.iterdir())
    # Crack500: image = .jpg, mask = same stem + .png
    image_files = [f for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')
                   and ':Zone.Identifier' not in f.name]

    if not image_files:
        # Some versions store as .png images too — distinguish by paired files
        png_files = [f for f in all_files if f.suffix.lower() == '.png'
                     and ':Zone.Identifier' not in f.name]
        # If .jpg exists for a stem → .png is mask. If no .jpg → .png is image.
        jpg_stems = {f.stem for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')}
        image_files = [f for f in png_files if f.stem not in jpg_stems]

    converted = 0
    skipped   = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find corresponding mask (.png with same stem)
        mask_path = crop_dir / f"{stem}.png"
        if not mask_path.exists():
            # Try .bmp
            mask_path = crop_dir / f"{stem}.bmp"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Convert mask to YOLO seg labels
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def write_dataset_yaml(dst: Path, num_train: int, num_val: int, num_test: int):
    """Write ultralytics-compatible dataset.yaml."""
    yaml_content = f"""# Crack500 — YOLO seg format
# Auto-generated by convert_crack500.py

path: {dst.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# train: ~{num_train} images
# val:   ~{num_val} images
# test:  ~{num_test} images
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")


def verify_conversion(dst: Path):
    """Quick sanity check on converted dataset."""
    print("\n[Verify] Checking converted dataset...")
    issues = 0
    for split in ["train", "val", "test"]:
        img_dir = dst / "images" / split
        lbl_dir = dst / "labels" / split
        if not img_dir.exists():
            continue

        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        lbls = list(lbl_dir.glob("*.txt"))

        # Check counts match
        if len(imgs) != len(lbls):
            print(f"  [!] {split}: {len(imgs)} images vs {len(lbls)} labels — mismatch!")
            issues += 1
        else:
            print(f"  {split}: {len(imgs)} images ✓")

        # Check a few labels are non-empty
        non_empty = sum(1 for l in lbls if l.stat().st_size > 0)
        empty     = len(lbls) - non_empty
        print(f"    labels with cracks: {non_empty} | empty (no crack): {empty}")

        if non_empty == 0:
            print(f"  [!] {split}: ALL labels are empty — check mask paths!")
            issues += 1

    if issues == 0:
        print("\n  ✓ Dataset looks good!")
    else:
        print(f"\n  ✗ {issues} issue(s) found — check output above.")

    return issues == 0


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        required=True,
        help="Path to crack500 root dir (contains traincrop/, valcrop/, testcrop/)"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default=None,
        help="Output directory (default: <src>_yolo)"
    )
    parser.add_argument(
        "--verify",
        action="store_true",
        default=True,
        help="Run sanity check after conversion"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve() if args.dst else src.parent / f"{src.name}_yolo"

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    counts = {}
    for split in ["train", "val", "test"]:
        n = process_split(
            src_dir    = src,
            dst_img_dir= dst / "images" / split,
            dst_lbl_dir= dst / "labels" / split,
            split_name = split,
        )
        counts[split] = n

    write_dataset_yaml(dst, counts["train"], counts["val"], counts["test"])

    if args.verify:
        verify_conversion(dst)

    print(f"\n[Done] Converted dataset at: {dst}")
    print(f"\nNext step — test YOLO11 loads it:")
    print(f"  from ultralytics import YOLO")
    print(f"  model = YOLO('yolo11n-seg.pt')")
    print(f"  model.train(data='{dst}/dataset.yaml', epochs=1, imgsz=512)")


if __name__ == "__main__":
    main()
# (appended — nothing, file is complete)


Writing scripts/convert_crack500.py


In [11]:
%%writefile scripts/convert_crack500_uncropped.py
#!/usr/bin/env python3
"""
Crack500 Uncropped Test/Val → YOLO seg format converter
======================================================
Converts the original uncropped validation and test sets of Crack500.
Handles EXIF orientation for images by rotating the corresponding masks.

Source directories:
  data/datasets/crack500/valdata/   ← contains {stem}.jpg and {stem}_mask.png
  data/datasets/crack500/testdata/  ← contains {stem}.jpg and {stem}_mask.png

Output (YOLO seg format):
  data/datasets/crack500_uncropped_yolo/
  ├── images/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── val/
  │   └── test/
  └── dataset.yaml
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm
from PIL import Image


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def get_exif_rotation(img_path: Path):
    """Retrieve EXIF orientation tag from image."""
    try:
        with Image.open(img_path) as im:
            exif = im.getexif()
            if exif:
                return exif.get(274)  # 274 is the Orientation tag
    except Exception:
        pass
    return None


def rotate_mask_to_match_image(mask: np.ndarray, exif_orientation: int) -> np.ndarray:
    """Rotate mask array to match image rotation applied by cv2.imread based on EXIF."""
    if exif_orientation == 6:
        return cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE)
    elif exif_orientation == 8:
        return cv2.rotate(mask, cv2.ROTATE_90_COUNTERCLOCKWISE)
    elif exif_orientation == 3:
        return cv2.rotate(mask, cv2.ROTATE_180)
    return mask


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int, exif_orientation: int = None) -> list[str]:
    """
    Read binary PNG mask → rotate based on EXIF → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    if exif_orientation:
        mask = rotate_mask_to_match_image(mask, exif_orientation)

    # Threshold (Crack500 masks are binary 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_dir: Path, split_name: str):
    """Process uncropped val or test split."""
    split_dir = src_dir / f"{split_name}data"
    if not split_dir.exists():
        print(f"  [Warning] Directory {split_dir} does not exist, skipping split {split_name}.")
        return 0

    dst_img_dir = dst_dir / "images" / split_name
    dst_lbl_dir = dst_dir / "labels" / split_name

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all image files (jpg/jpeg/png that do not contain '_mask')
    all_files = sorted(split_dir.iterdir())
    image_files = [
        f for f in all_files 
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and '_mask' not in f.name.lower()
        and ':Zone.Identifier' not in f.name
    ]

    converted = 0
    skipped = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find mask (stem + "_mask.png")
        mask_path = split_dir / f"{stem}_mask.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions (matches how cv2.imread auto-rotates it based on EXIF)
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Get EXIF rotation from image
        exif_orientation = get_exif_rotation(img_path)

        # Convert mask to YOLO seg labels (rotating it to match)
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h, exif_orientation)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 Uncropped splits to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        default="data/datasets/crack500",
        help="Path to crack500 root dir"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default="data/datasets/crack500_uncropped_yolo",
        help="Output directory"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve()

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    if dst.exists():
        print(f"[Warning] Output directory exists, clearing: {dst}")
        shutil.rmtree(dst)

    dst.mkdir(parents=True, exist_ok=True)

    counts = {}
    for split in ["val", "test"]:
        n = process_split(src, dst, split)
        counts[split] = n

    # Write dataset_uncropped.yaml
    yaml_content = f"""# Crack500 Uncropped — YOLO seg format
# Auto-generated by convert_crack500_uncropped.py

path: {dst.resolve()}
train: images/val
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# val:   ~{counts.get('val', 0)} images (uncropped)
# test:  ~{counts.get('test', 0)} images (uncropped)
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")
    print(f"[Done] Converted uncropped splits successfully.")


if __name__ == "__main__":
    main()


Writing scripts/convert_crack500_uncropped.py


In [12]:
%%writefile scripts/generate_teacher_logits.py
#!/usr/bin/env python3
"""
Generate SAM 2 teacher logits for crack500 training images.
Run ONCE before training. Saves .npy logit files to data/teacher_logits/

Usage (from ~/distill):
  python scripts/generate_teacher_logits.py           # full dataset
  python scripts/generate_teacher_logits.py --resume  # skip already done
  python scripts/generate_teacher_logits.py --max 50  # only first N images
"""

import argparse
import sys
import os
import cv2
import numpy as np
import torch
from pathlib import Path

# ── project root on path ─────────────────────────────────────
ROOT = Path(__file__).parent.parent.resolve()
sys.path.insert(0, str(ROOT))

# ── fixed paths (relative to ~/distill) ─────────────────────
DATASET_DIR = ROOT / "data/datasets/crack500"
LOGITS_DIR  = ROOT / "data/teacher_logits"
SAM2_CKPT   = ROOT / "checkpoints/sam2_hiera_large.pt"
SAM2_CFG    = "configs/sam2.1/sam2.1_hiera_l.yaml"


def mask_to_bbox(mask: np.ndarray):
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any():
        return None
    r1, r2 = np.where(rows)[0][[0, -1]]
    c1, c2 = np.where(cols)[0][[0, -1]]
    return np.array([c1, r1, c2, r2], dtype=np.float32)


def mask_to_centroid(mask: np.ndarray):
    M = cv2.moments(mask)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
        h, w = mask.shape[:2]
        if 0 <= cX < w and 0 <= cY < h and mask[cY, cX] > 0:
            return np.array([[cX, cY]], dtype=np.float32)
    # Fallback to maximum of distance transform (guaranteed to be inside mask)
    dist_transform = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    _, _, _, max_loc = cv2.minMaxLoc(dist_transform)
    cX, cY = max_loc
    return np.array([[cX, cY]], dtype=np.float32)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--resume", action="store_true",
                        help="Skip images that already have logits")
    parser.add_argument("--max", type=int, default=None,
                        help="Max images to process (for testing)")
    parser.add_argument("--img-dir", type=str, default=None,
                        help="Path to directory of training images")
    parser.add_argument("--mask-dir", type=str, default=None,
                        help="Path to directory of training masks")
    parser.add_argument("--prefix", type=str, default="crack500_",
                        help="Prefix to use for saved logits files")
    parser.add_argument("--prompt-type", type=str, default="box_centroid",
                        choices=["box", "box_centroid"],
                        help="Prompt type to use (box or box_centroid)")
    parser.add_argument("--logits-dir", type=str, default=None,
                        help="Override output logits directory")
    parser.add_argument("--dataset", type=str, default=None,
                        help="Path to YOLO dataset or raw dataset directory")
    parser.add_argument("--sam-ckpt", type=str, default=str(SAM2_CKPT),
                        help="Path to SAM 2 checkpoint file")
    parser.add_argument("--sam-cfg", type=str, default=SAM2_CFG,
                        help="SAM 2 model config name or path")
    args = parser.parse_args()

    if args.dataset and not args.img_dir:
        ds_path = Path(args.dataset).expanduser().resolve()
        if (ds_path / "images/train").exists():
            args.img_dir = str(ds_path / "images/train")
            args.mask_dir = str(ds_path / "masks/train") if (ds_path / "masks/train").exists() else str(ds_path / "images/train")
        elif ds_path.exists():
            args.img_dir = str(ds_path)
            args.mask_dir = str(ds_path)

    if args.logits_dir:
        logits_dir = Path(args.logits_dir).expanduser().resolve()
    else:
        logits_dir = ROOT / "data/teacher_logits"
        
    orig_logits_dir = logits_dir
    # On Kaggle, redirect to /tmp/ to avoid exceeding 20GB disk limit
    if "/kaggle/" in str(logits_dir):
        logits_dir = Path("/tmp") / logits_dir.name
        logits_dir.mkdir(parents=True, exist_ok=True)
        try:
            if os.path.lexists(orig_logits_dir):
                if os.path.islink(orig_logits_dir):
                    os.unlink(orig_logits_dir)
                elif orig_logits_dir.is_dir() and not list(orig_logits_dir.glob("*.npy")):
                    orig_logits_dir.rmdir()
            if not orig_logits_dir.exists():
                orig_logits_dir.parent.mkdir(parents=True, exist_ok=True)
                os.symlink(logits_dir, orig_logits_dir)
                print(f"[Logits] Created symlink: {orig_logits_dir} -> {logits_dir}")
        except Exception as e:
            print(f"[Logits Warning] Could not symlink {orig_logits_dir} -> {logits_dir}: {e}")
        
    logits_dir.mkdir(parents=True, exist_ok=True)

    if args.img_dir:
        # Single custom dataset mode
        datasets = [{
            "name": "Custom",
            "img_dir": Path(args.img_dir).expanduser().resolve(),
            "mask_dir": Path(args.mask_dir).expanduser().resolve() if args.mask_dir else Path(args.img_dir).expanduser().resolve(),
            "prefix": args.prefix
        }]
    else:
        # Multi-dataset auto-mode
        datasets = [
            {
                "name": "Crack500",
                "img_dir": ROOT / "data/datasets/crack500/traincrop",
                "mask_dir": ROOT / "data/datasets/crack500/traincrop",
                "prefix": "crack500_"
            },
            {
                "name": "DeepCrack",
                "img_dir": ROOT / "data/datasets/deepcrack/train_img",
                "mask_dir": ROOT / "data/datasets/deepcrack/train_lab",
                "prefix": "deepcrack_"
            }
        ]

    # Filter out non-existent datasets
    active_datasets = []
    for d in datasets:
        if d["img_dir"].exists():
            active_datasets.append(d)
        else:
            if args.img_dir:
                print(f"ERROR: Image directory not found: {d['img_dir']}")
                return
            else:
                print(f"Skipping {d['name']} logits generation: directory {d['img_dir']} does not exist.")

    if not active_datasets:
        print("No active datasets to process. Exiting.")
        return

    # ── Load SAM 2 ───────────────────────────────────────────
    ckpt_path = Path(args.sam_ckpt)
    if not ckpt_path.exists():
        # Search alternate candidate checkpoint locations
        for cand in [ROOT / "checkpoints/sam2.1_hiera_large.pt", ROOT / "checkpoints/sam2_hiera_large.pt"]:
            if cand.exists():
                ckpt_path = cand
                break
    print(f"Loading SAM 2 from {ckpt_path}...")
    predictor = None
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Strategy A: build_sam2 with candidate Hydra config paths
    try:
        from sam2.build_sam import build_sam2
        from sam2.sam2_image_predictor import SAM2ImagePredictor

        cfg_candidates = [
            args.sam_cfg,
            "sam2.1/sam2.1_hiera_l.yaml",
            "configs/sam2.1/sam2.1_hiera_l.yaml",
            "sam2/configs/sam2.1/sam2.1_hiera_l.yaml",
            "sam2_hiera_l.yaml",
            "configs/sam2/sam2_hiera_l.yaml"
        ]
        if ckpt_path.exists():
            for cfg in cfg_candidates:
                try:
                    model = build_sam2(cfg, str(ckpt_path), device=device)
                    predictor = SAM2ImagePredictor(model)
                    print(f"SAM 2 loaded successfully via build_sam2 (config: {cfg}) on {device} ✓\n")
                    break
                except Exception:
                    continue
    except Exception as e:
        print(f"[SAM2 Init Warning] build_sam2 failed: {e}")

    # Strategy B: SAM2ImagePredictor from HuggingFace
    if predictor is None:
        try:
            from sam2.sam2_image_predictor import SAM2ImagePredictor
            print("Attempting to load SAM 2 from HuggingFace Hub...")
            predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2.1-hiera-large", device=device)
            print(f"SAM 2 loaded from HuggingFace on {device} ✓\n")
        except Exception as e:
            print(f"[SAM2 Init Warning] HuggingFace load failed: {e}")

    if predictor is None:
        raise RuntimeError("FATAL: Could not initialize SAM 2! Check torch, cuda, checkpoint, and sam2 installation.")

    for d in active_datasets:
        img_dir = d["img_dir"]
        mask_dir = d["mask_dir"]
        prefix = d["prefix"]

        # ── Process images ───────────────────────────────────────
        # Check for jpg/jpeg files first to avoid loading PNG masks as images in Crack500 traincrop
        images = sorted([
            f for f in img_dir.iterdir()
            if f.suffix.lower() in ('.jpg', '.jpeg')
            and ':Zone.Identifier' not in f.name
        ])
        if not images:
            images = sorted([
                f for f in img_dir.iterdir()
                if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
                and ':Zone.Identifier' not in f.name
            ])
        if args.max:
            images = images[:args.max]

        print(f"\n>>> Processing {d['name']} ({len(images)} images) <<<")
        print(f"  Images: {img_dir}")
        print(f"  Masks:  {mask_dir}")
        print(f"  Prefix: {prefix}\n")

        generated = 0
        skipped   = 0
        failed    = 0

        for img_path in images:
            image_id   = f"{prefix}{img_path.stem}"
            logit_file = logits_dir / f"{image_id}_logits.npy"

            # Skip if already done
            if args.resume and logit_file.exists():
                skipped += 1
                continue

            # Load image
            image = cv2.imread(str(img_path))
            if image is None:
                failed += 1
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Load instance masks from binary PNG/BMP file or YOLO label .txt file
            instance_masks = []

            mask_path = mask_dir / f"{img_path.stem}.png"
            if not mask_path.exists():
                mask_path = mask_dir / f"{img_path.stem}.bmp"
            
            # Check raw dataset fallback paths (e.g. train_lab / traincrop)
            if not mask_path.exists():
                for alt_name in ["train_lab", "traincrop", "masks"]:
                    alt_path = img_dir.parent.parent / alt_name / f"{img_path.stem}.png"
                    if alt_path.exists():
                        mask_path = alt_path
                        break

            # Check YOLO txt label fallback
            label_txt_path = img_dir.parent.parent / "labels" / img_dir.name / f"{img_path.stem}.txt"

            if mask_path.exists():
                binary = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                if binary is not None:
                    binary = (binary > 127).astype(np.uint8)
                    num_labels, labels_map = cv2.connectedComponents(binary)
                    for label_id in range(1, num_labels):
                        inst = (labels_map == label_id).astype(np.uint8)
                        if inst.sum() >= 50:
                            instance_masks.append(inst)
            elif label_txt_path.exists():
                img_h, img_w = image.shape[:2]
                with open(label_txt_path, "r") as lf:
                    lines = lf.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) < 7:  # class + at least 3 (x,y) pairs
                        continue
                    try:
                        coords = [float(x) for x in parts[1:]]
                        pts = np.array(coords).reshape(-1, 2)
                        pts[:, 0] *= img_w
                        pts[:, 1] *= img_h
                        inst = np.zeros((img_h, img_w), dtype=np.uint8)
                        cv2.fillPoly(inst, [pts.astype(np.int32)], 1)
                        if inst.sum() >= 50:
                            instance_masks.append(inst)
                    except Exception:
                        pass

            if not instance_masks:
                failed += 1
                continue

            all_logits = []
            predictor.set_image(image)

            # Capture encoder features
            features = predictor._features
            image_embed = features["image_embed"].cpu().half().numpy()
            feat0 = features["high_res_feats"][0].cpu().half().numpy()
            feat1 = features["high_res_feats"][1].cpu().half().numpy()

            for instance in instance_masks:
                box = mask_to_bbox(instance)
                if box is None:
                    continue

                if args.prompt_type == "box_centroid":
                    centroid = mask_to_centroid(instance)
                    point_labels = np.array([1], dtype=np.int32)
                    with torch.no_grad():
                        _, _, logits = predictor.predict(
                            point_coords=centroid,
                            point_labels=point_labels,
                            box=box,
                            multimask_output=False,
                        )
                else:
                    with torch.no_grad():
                        _, _, logits = predictor.predict(
                            box=box,
                            multimask_output=False,
                        )
                all_logits.append(logits[0].astype(np.float32))  # (256, 256)

            if all_logits:
                np.save(str(logit_file), np.stack(all_logits, axis=0))
                
                # Save encoder features in a shared features directory to save disk space
                features_dir = logits_dir.parent / "teacher_features"
                features_dir.mkdir(parents=True, exist_ok=True)
                feat_file = features_dir / f"{image_id}_features.npz"
                if not feat_file.exists():
                    np.savez_compressed(
                        str(feat_file),
                        image_embed=image_embed,
                        feat0=feat0,
                        feat1=feat1
                    )
                generated += 1
            else:
                failed += 1

            total = generated + skipped + failed
            if total % 50 == 0 or total == len(images):
                print(f"  {total}/{len(images)} | "
                      f"generated={generated} skipped={skipped} failed={failed}")

        print(f"\n  Finished {d['name']}: generated={generated}, skipped={skipped}, failed={failed}")

    print("\n[Done] All active datasets processed successfully.")
    print("Next step:")
    print("  python scripts/run_experiments.py --exp full_kd")


if __name__ == "__main__":
    main()


Writing scripts/generate_teacher_logits.py


In [13]:
# ── Step 1: Link Kaggle Inputs (Dataset & Teacher Logits) ──
import os, shutil
from pathlib import Path

input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists():
    input_dir = Path("/kaggle/input")

datasets_dir = Path("data/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir = Path("checkpoints")
checkpoints_dir.mkdir(parents=True, exist_ok=True)

# 1. Link Crack500. Order-independent: raw Crack500 (a folder holding traincrop/) wins wherever it is;
#    otherwise only a converted folder named exactly crack500_yolo. Other datasets in the input
#    (combined_yolo, crack500_yolo_augmented, deepcrack_yolo, ...) are never picked.
walk = list(os.walk(str(input_dir)))
raw_dirs = sorted(Path(r) for r, d, f in walk if "traincrop" in d)
yolo_dirs = sorted(Path(r) for r, d, f in walk if Path(r).name == "crack500_yolo" and "dataset.yaml" in f)
found_dataset = False
if raw_dirs or yolo_dirs:
    src, dest = (raw_dirs[0], datasets_dir / "crack500") if raw_dirs else (yolo_dirs[0], datasets_dir / "crack500_yolo")
    if len(raw_dirs or yolo_dirs) > 1:
        print(f"[Dataset Warning] {len(raw_dirs or yolo_dirs)} candidates found, using {src}")
    if os.path.lexists(dest):
        os.unlink(dest) if os.path.islink(dest) else shutil.rmtree(dest)
    os.symlink(src, dest)
    print(f"[Dataset] Linked {'raw Crack500' if raw_dirs else 'converted crack500_yolo'}: {src} -> {dest}")
    found_dataset = True
else:
    print("[Dataset Warning] Neither raw Crack500 (traincrop/) nor a crack500_yolo folder found in input.")

# 2. Link precomputed teacher logits
found_logits = False
target_dirs = ['teacher_logits_box', 'teacher_logits_box', 'teacher_logits']

for root, dirs, files in os.walk(str(input_dir)):
    root_p = Path(root)
    for target_name in target_dirs:
        if target_name in dirs:
            src_f = root_p / target_name
            dst_f = Path("data/teacher_logits_box")
            if os.path.lexists(dst_f):
                os.unlink(dst_f) if os.path.islink(dst_f) else shutil.rmtree(dst_f)
            os.symlink(src_f, dst_f)
            print(f"[Logits] Linked precomputed logits: {src_f} -> {dst_f}")
            found_logits = True
            break
    if found_logits:
        break
if not found_logits:
    print("[Logits Notice] Precomputed logits directory not found in input; will generate if needed.")


[Dataset] Linked raw Crack500: /kaggle/input/datasets/rauffatali/distill-datasetforme/datasets/crack500 -> data/datasets/crack500
[Logits] Linked precomputed logits: /kaggle/input/datasets/rauffatali/distill-datasetforme/teacher_logits -> data/teacher_logits_box


In [14]:
# ── Step 2: Convert Datasets & Verify Teacher Logits ──
# 1. Convert Cropped Crack500 if raw was linked and yolo not yet generated
if Path("data/datasets/crack500").exists() and not Path("data/datasets/crack500_yolo/dataset.yaml").exists():
    !python scripts/convert_crack500.py --src data/datasets/crack500 --dst data/datasets/crack500_yolo

# 2. Convert Uncropped Crack500 for OOD Evaluation
if Path("data/datasets/crack500/valdata").exists() and not Path("data/datasets/crack500_uncropped_yolo/dataset.yaml").exists():
    !python scripts/convert_crack500_uncropped.py --src data/datasets/crack500 --dst data/datasets/crack500_uncropped_yolo

# Canonical Crack500 split (1896 / 348 / 1124): anything else means the wrong dataset was linked
split_sizes = {s: len(list(Path(f"data/datasets/crack500_yolo/images/{s}").glob("*"))) for s in ("train", "val", "test")}
print(f"[Dataset] crack500_yolo split sizes: {split_sizes}")
assert split_sizes == {"train": 1896, "val": 348, "test": 1124}, f"[FATAL] Not the canonical Crack500 split: {split_sizes}"

# 3. Verify Logits
logits_dir = Path("data/teacher_logits_box")
logits_count = len(list(logits_dir.glob("*_logits.npy"))) if logits_dir.exists() else 0
print(f"[Verification] Found {logits_count} teacher logit files in {logits_dir}")

if logits_count == 0:
    print("=== Pre-computed logits not found in input; Generating SAM 2 Logits on GPU ===")
    ckpt_file = checkpoints_dir / "sam2_hiera_large.pt"
    if not ckpt_file.exists():
        !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt -O checkpoints/sam2_hiera_large.pt
    !pip install -q SAM-2 || pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
    !python scripts/generate_teacher_logits.py --prompt-type box --logits-dir data/teacher_logits_box --dataset data/datasets/crack500_yolo

logits_count = len(list(logits_dir.glob("*_logits.npy")))
assert logits_count > 0, f"[FATAL ERROR] 0 logit files found in {logits_dir}. KD training cannot proceed without teacher supervision!"
print(f"[Verification Passed] Ready to train with {logits_count} real SAM 2 teacher logits from {logits_dir}!")


[Convert] Source: /kaggle/input/datasets/rauffatali/distill-datasetforme/datasets/crack500
[Convert] Output: /kaggle/working/data/datasets/crack500_yolo

  train: 1896 images converted, 0 skipped
  val: 348 images converted, 0 skipped
  test: 1124 images converted, 0 skipped

  dataset.yaml written to /kaggle/working/data/datasets/crack500_yolo/dataset.yaml

[Verify] Checking converted dataset...
  train: 1896 images ✓
    labels with cracks: 1896 | empty (no crack): 0
  val: 348 images ✓
    labels with cracks: 348 | empty (no crack): 0
  test: 1124 images ✓
    labels with cracks: 1124 | empty (no crack): 0

  ✓ Dataset looks good!

[Done] Converted dataset at: /kaggle/working/data/datasets/crack500_yolo

Next step — test YOLO11 loads it:
  from ultralytics import YOLO
  model = YOLO('yolo11n-seg.pt')
  model.train(data='/kaggle/working/data/datasets/crack500_yolo/dataset.yaml', epochs=1, imgsz=512)
[Convert] Source: /kaggle/input/datasets/rauffatali/distill-datasetforme/datasets/cra

In [15]:
# ── Step 3: Run Full Production Training (mask_kd_seed0_150ep) ──
import sys
sys.path.insert(0, ".")
from pathlib import Path
from distillation.kd_trainer import KDSegmentationTrainer
from utils.config_loader import load_config, override_config

cfg = load_config("configs/config.yaml")
EXPERIMENT_NAME = "mask_kd_seed0_150ep"

overrides = {
    'distillation.enabled': True,
    'distillation.temperature': 3.7769,
    'distillation.progressive.enabled': False,
    'distillation.losses.task.weight': 1.0,
    'distillation.losses.mask_kd.enabled': True,
    'distillation.losses.mask_kd.weight': 0.9612,
    'distillation.losses.mask_kd.focused': False,
    'distillation.losses.mask_kd.high_res': False,
    'distillation.losses.feature.enabled': False,
    'distillation.losses.boundary.enabled': False,
    'distillation.losses.affinity.enabled': False,
    'distillation.losses.tversky.enabled': False,
    'train.epochs': 150,
    'train.amp': False,
}
overrides["project.name"] = "crack_distill"
overrides["project.experiment"] = EXPERIMENT_NAME
overrides["project.seed"] = 0
overrides["data.datasets"] = [{"name": "crack500", "path": "data/datasets/crack500_yolo", "format": "yolo"}]
overrides["teacher.logits_dir"] = "data/teacher_logits_box/"
if "train.epochs" not in overrides:
    overrides["train.epochs"] = 150
if "train.amp" not in overrides:
    overrides["train.amp"] = False

cfg = override_config(cfg, overrides)

print(f"=== Starting Run: {EXPERIMENT_NAME} ===")
print("Config in effect: Seed=0, Epochs=150, AMP=False, Freezing=False")

trainer = KDSegmentationTrainer(cfg)
trainer.train()
print("✓ Training completed successfully!")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
=== Starting Run: mask_kd_seed0_150ep ===
Config in effect: Seed=0, Epochs=150, AMP=False, Freezing=False
[Augmentation Policy] Spatial augmentations (mosaic, affine, flip, erasing) disabled for controlled comparison (allow_spatial_aug=False). Seed: 0
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/datasets/crack500_yolo/dataset.yaml, degrees=0.0, det

In [16]:
# ── Step 4: Validate Best Checkpoint & Export Results ──
import json, os
from pathlib import Path
from ultralytics import YOLO
from utils.checkpoint import resolve_checkpoint, get_checkpoint_manifest, save_checkpoint_manifest

# Deterministically resolve checkpoint without fuzzy glob collision
candidate_ckpt = getattr(trainer, "best", None) or Path("runs") / "crack_distill" / EXPERIMENT_NAME / "weights" / "best.pt"
best_pt_path = resolve_checkpoint(candidate_ckpt, expected_experiment=EXPERIMENT_NAME)
manifest = get_checkpoint_manifest(best_pt_path, experiment_name=EXPERIMENT_NAME, seed=0)
manifest_file = save_checkpoint_manifest(manifest)

print(f"Evaluating verified checkpoint: {best_pt_path}")
print(f"Checkpoint SHA256: {manifest['sha256']}")
model = YOLO(str(best_pt_path))

# 1. Validate on Crack500 In-Domain Val Set
print("\n--- In-Domain Cropped Validation ---")
val_metrics = model.val(data="data/datasets/crack500_yolo/dataset.yaml", split="val", verbose=True)

results = {
    "experiment": "mask_kd_seed0_150ep",
    "arm": "mask_kd",
    "seed": 0,
    "checkpoint": str(best_pt_path),
    "checkpoint_sha256": manifest["sha256"],
    "metrics_indomain": {
        "mask_mAP50": float(val_metrics.seg.map50),
        "mask_mAP50_95": float(val_metrics.seg.map),
        "box_mAP50": float(val_metrics.box.map50),
        "box_mAP50_95": float(val_metrics.box.map),
        "mask_precision": float(val_metrics.seg.p[0]) if hasattr(val_metrics.seg, 'p') and len(val_metrics.seg.p) > 0 else float(val_metrics.seg.mp),
        "mask_recall": float(val_metrics.seg.r[0]) if hasattr(val_metrics.seg, 'r') and len(val_metrics.seg.r) > 0 else float(val_metrics.seg.mr)
    }
}

# 2. Validate on Crack500 Uncropped (OOD) Set
uncropped_yaml = Path("data/datasets/crack500_uncropped_yolo/dataset.yaml")
if uncropped_yaml.exists():
    print("\n--- Out-of-Distribution (Uncropped) Validation ---")
    ood_metrics = model.val(data=str(uncropped_yaml), split="val", verbose=True)
    results["metrics_ood"] = {
        "ood_mask_mAP50": float(ood_metrics.seg.map50),
        "ood_mask_mAP50_95": float(ood_metrics.seg.map),
        "ood_box_mAP50": float(ood_metrics.box.map50),
        "ood_box_mAP50_95": float(ood_metrics.box.map)
    }
else:
    print("Warning: Uncropped dataset.yaml not found — skipping OOD evaluation.")

print("\n" + "="*60)
print(f"🎯 FINAL EVALUATION SUMMARY (mask_kd_seed0_150ep):")
print(f"   In-Domain Mask mAP50    : {results['metrics_indomain']['mask_mAP50']:.4f}")
print(f"   In-Domain Mask mAP50-95 : {results['metrics_indomain']['mask_mAP50_95']:.4f}")
print(f"   In-Domain Box mAP50     : {results['metrics_indomain']['box_mAP50']:.4f}")
if "metrics_ood" in results:
    print(f"   OOD Uncropped Mask mAP50    : {results['metrics_ood']['ood_mask_mAP50']:.4f}")
    print(f"   OOD Uncropped Mask mAP50-95 : {results['metrics_ood']['ood_mask_mAP50_95']:.4f}")
print("="*60)

out_file = Path(f"/kaggle/working/results/mask_kd_seed0_150ep.json")
out_file.parent.mkdir(parents=True, exist_ok=True)
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(f"Saved structured summary to {out_file}")


Evaluating verified checkpoint: /kaggle/working/runs/segment/crack_distill/mask_kd_seed0_150ep/weights/best.pt
Checkpoint SHA256: 04bf134cb2c1e8e4cad949272d0dff901bf8462c2fd617f9ae8205f5d4051a75

--- In-Domain Cropped Validation ---
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n-seg summary (fused): 129 layers, 2,944,651 parameters, 0 gradients, 10.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1407.8±338.9 MB/s, size: 58.9 KB)
val: Scanning /kaggle/working/data/datasets/crack500_yolo/labels/val.cache... 348 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 348/348 132.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 5.7it/s 3.8s
                   all        348        630       0.69      0.441      0.495      0.283      0.669      0.437      0.462      0.152
Speed: 0.6ms preprocess, 3.0ms inference, 0.0ms lo